# encode_ordinance.ipynb
# Zoning Ordinance → `zoning_district_standards.csv` + `wu_transect_standards.csv`

Parses the Phoenix Zoning Ordinance HTML into two machine-readable CSVs:

1. **`zoning_district_standards.csv`** — one row per zoning district (residential, commercial,
   overlay, DTC character areas, special districts) with all development standards.
2. **`wu_transect_standards.csv`** — one row per WU Code transect district (T3:2 through SD)
   with height, coverage, setback, use, and parking standards from Chapter 13.

**Data source:** `data/reference/zoning_ordinance_webpage.html`  
**Run order:** Execute cells top-to-bottom. Manual review checkpoints are marked ⚠️.

**Column additions vs. initial version:**
- `adu_coverage_bonus_pct` — extra 10% coverage allowed for ADU/shade structures (§608)
- `max_height_stories_prd` — PRD option story count
- `max_lot_coverage_pct_prd` — PRD option lot coverage
- `dtc_flag`, `dtc_character_area` — Downtown Code identification
- `mf_conversion_flag` — §711 multi-family conversion eligibility
- `ri_overlay_eligible`, `hr_overlay_eligible` — overlay applicability flags

*AI use disclosure: This notebook was generated with AI assistance (Claude, Anthropic)
as part of the GIS 322 Phoenix Upzoning Scanner project. All values verified against
the Phoenix Zoning Ordinance HTML source. Manual review checkpoints are provided.*

## Section 1 — Setup

In [70]:
from pathlib import Path
import re
import pandas as pd
from bs4 import BeautifulSoup

PROJECT_ROOT = Path(r"D:\DevSource\phoenix-zoning-scanner")
DATA_DIR = PROJECT_ROOT / "data"

# ── Inputs ────────────────────────────────────────────────────────────────────
ORDINANCE_HTML = DATA_DIR / "reference" / "zoning_ordinance_webpage.html"

# ── Outputs ───────────────────────────────────────────────────────────────────
OUTPUT_CSV    = DATA_DIR / "reference" / "zoning_district_standards.csv"
OUTPUT_WU_CSV = DATA_DIR / "reference" / "wu_transect_standards.csv"

print("Ordinance HTML:", ORDINANCE_HTML)
print("Output CSV:    ", OUTPUT_CSV)
print("Output WU CSV: ", OUTPUT_WU_CSV)

Ordinance HTML: D:\DevSource\phoenix-zoning-scanner\data\reference\zoning_ordinance_webpage.html
Output CSV:     D:\DevSource\phoenix-zoning-scanner\data\reference\zoning_district_standards.csv
Output WU CSV:  D:\DevSource\phoenix-zoning-scanner\data\reference\wu_transect_standards.csv


In [71]:
# Load and parse the HTML once — this takes ~10–15 seconds for a 5.8 MB file
with open(ORDINANCE_HTML, "r", encoding="utf-8") as f:
    soup = BeautifulSoup(f, "html.parser")

print("Parsed. Title:", soup.title.get_text(strip=True))

Parsed. Title: Your Selections | Phoenix Zoning Ordinance


## Section 2 — Parse structured table districts (Sections 609–618)

Districts RE-35 through R-5 each have a `Section X.B` table with two columns:
**(A) Standard Option** and **(B) Planned Residential Development (PRD)**.

Both columns are now extracted. PRD values are stored in `_prd` columns for
reference — they are not used in the primary feasibility scoring, but support
the density bonus and development option switching logic in the scanner.

In [72]:
# District registry — (section_id, district_code, district_name)
# Only districts whose Section B contains a structured HTML table go here.
TABLE_DISTRICTS = [
    ("ZO_609.B", "RE-35",  "RE-35 Single-Family Residence"),
    ("ZO_610.B", "R1-18",  "R1-18 Single-Family Residence"),
    ("ZO_611.B", "R1-10",  "R1-10 Single-Family Residence"),
    ("ZO_612.B", "R1-8",   "R1-8 Single-Family Residence"),
    ("ZO_613.B", "R1-6",   "R1-6 Single-Family Residence"),
    ("ZO_614.B", "R-2",    "R-2 Multi-Family Residence"),
    ("ZO_615.B", "R-3",    "R-3 Multi-Family Residence"),
    ("ZO_616.B", "R-3A",   "R-3A Multi-Family Residence"),
    ("ZO_617.B", "R-4",    "R-4 Multi-Family Residence"),
    ("ZO_618.B", "R-5",    "R-5 Multi-Family Residence — Restricted Commercial"),
]

In [73]:
def parse_table_district(section_id: str) -> dict:
    """
    Parse a structured B-table district section from the ordinance HTML.
    Returns a dict with raw cell text for each row label.
    
    Row mapping (from Section 609.B onwards):
      Row (1)  → lot_width
      Row (2)  → lot_depth
      Row (3)  → density (Standard Option)
      Row (5)  → setbacks (Standard Option — needs further parsing)
      Row (10) → height
      Row (11) → lot_coverage
    """
    sec = soup.find("section", id=section_id)
    if not sec:
        raise ValueError(f"Section {section_id!r} not found in HTML")
    
    tables = sec.find_all("table")
    if not tables:
        raise ValueError(f"No table found in {section_id!r}")
    
    table = tables[0]
    rows = table.find_all("tr")
    
    raw = {}
    for row in rows:
        cells = row.find_all(["td", "th"])
        if len(cells) < 2:
            continue
        # Row number is in cell 0, label in cell 1, standard in cell 2, PRD in cell 3
        # (header row has no row number)
        if len(cells) >= 3:
            row_num = cells[0].get_text(strip=True)
            label   = cells[1].get_text(strip=True)
            std_val = cells[2].get_text(strip=True)
            prd_val = cells[3].get_text(strip=True) if len(cells) > 3 else ""
        else:
            row_num = ""
            label   = cells[0].get_text(strip=True)
            std_val = cells[1].get_text(strip=True)
            prd_val = ""
        
        key = row_num.strip("()")
        raw[key] = {
            "label": label,
            "standard": std_val,
            "prd": prd_val,
        }
    
    return raw


# ── Test on RE-35 ─────────────────────────────────────────────────────────────
test_raw = parse_table_district("ZO_609.B")
for k, v in test_raw.items():
    print(f"  [{k}] {v['label'][:40]:<40} STD={v['standard'][:35]!r}")

  [Category] (A)Standard Option                       STD='(B)Planned Residential Development('
  [1] Lot width(minimum)                       STD='150 feet'
  [2] Lotdepth (minimum)                       STD='175 feet'
  [3] Developmentdensity(maximum)              STD='1.2PDU/AC (gross)'
  [4] Subdividedlots(maximum)                  STD='1.2lots/AC (gross)'
  [5] Individuallotsetbacks(minimum)           STD='Front: 40 feetRear: 40 feetSides: 2'
  [6] Garage door/carport entry setback(minimu STD='Same as individuallotsetbacks'
  [7] Projections                              STD='Per Section701.A.3.a'
  [8] Developmentperimeterbuildingsetback(mini STD='None'
  [9] Perimeterstreetlandscape setback(minimum STD='None'
  [10] Building height(maximum)                 STD='2storiesand 30 feet'
  [11] Lot coverage(maximum)                    STD='25%, except if allstructuresare les'
  [12] Common open space(minimum)               STD='None'
  [13] Street frontage requirement              STD=

In [74]:
def extract_ft(text: str) -> float | None:
    """Extract the first integer/decimal foot value. Returns None for missing/None."""
    if not text or re.search(r'\bNone\b', text, re.I):
        return None
    m = re.search(r'([\d]+\.?[\d]*)', text)
    return float(m.group(1)) if m else None


def extract_height(text: str) -> tuple[float | None, int | None]:
    """Extract (max_ft, max_stories) from a height cell string."""
    stories_m = re.search(r'(\d+)\s*stor', text, re.I)
    feet_m    = re.search(r'(\d+)\s*feet', text, re.I)
    stories = int(stories_m.group(1)) if stories_m else None
    feet    = float(feet_m.group(1))  if feet_m    else None
    return feet, stories


def extract_setbacks(text: str) -> dict:
    """
    Extract Standard Option setback values from a setback cell string.
    Handles symmetric ('Sides: X feet') and asymmetric
    ('Interior sides: X feet and Y feet') patterns.
    """
    def grab(pattern: str, t: str) -> float | None:
        m = re.search(pattern + r'[:\s]+(\d+)\s*feet', t, re.I)
        return float(m.group(1)) if m else None

    front       = grab(r'Front', text)
    rear        = grab(r'Rear', text)
    street_side = grab(r'Street side', text)

    sides_sym_m  = re.search(r'Sides:\s*(\d+)\s*feet', text, re.I)
    sides_asym_m = re.search(
        r'Interior sides[:\s]+(\d+)\s*feet\s*and\s*(\d+)\s*feet', text, re.I
    )

    if sides_asym_m:
        int_side_a = float(sides_asym_m.group(1))
        int_side_b = float(sides_asym_m.group(2))
    elif sides_sym_m:
        int_side_a = float(sides_sym_m.group(1))
        int_side_b = float(sides_sym_m.group(1))
    else:
        int_side_a = None
        int_side_b = None

    return {
        "min_front_setback_ft":           front,
        "min_rear_setback_ft":            rear,
        "min_street_side_setback_ft":     street_side,
        "min_interior_side_setback_a_ft": int_side_a,
        "min_interior_side_setback_b_ft": int_side_b,
    }


def extract_lot_coverage(text: str) -> float | None:
    """
    Extract the base Standard Option lot coverage %.
    '50%, plus an additional 10% for an ADU...' → 50.0
    '30%, plus an additional 10% for an ADU...' → 30.0
    '25%, except if all structures...'           → 25.0
    """
    m = re.search(r'^(\d+)\s*%', text.strip())
    return float(m.group(1)) if m else None


def extract_adu_coverage_bonus(text: str) -> float | None:
    """
    Extract the ADU/shade-structure lot coverage bonus %.
    'plus an additional 10% for an ADU and/or attached shade structures' → 10.0
    Returns None when no bonus language is present.
    """
    m = re.search(r'additional\s+(\d+)\s*%\s*for\s+an\s+ADU', text, re.I)
    return float(m.group(1)) if m else None


def extract_prd_coverage(text: str) -> float | None:
    """
    Extract PRD option lot coverage.
    '60% total for development' → 60.0
    '40% total for development' → 40.0
    """
    m = re.search(r'^(\d+)\s*%', text.strip())
    return float(m.group(1)) if m else None


# ── Sanity tests ──────────────────────────────────────────────────────────────
assert extract_ft("150 feet") == 150.0
assert extract_ft("None.")    is None
assert extract_height("2 stories and 30 feet") == (30.0, 2)
assert extract_height("4 stories and 48 feet") == (48.0, 4)
sb = extract_setbacks("Front: 20 feetRear: 25 feetStreet side: 10 feetInterior sides: 10 feet and 3 feet")
assert sb["min_front_setback_ft"] == 20.0
assert sb["min_interior_side_setback_b_ft"] == 3.0
cov_text = "50%, plus an additional 10% for an ADU and/or attached shade structures Total: 60%"
assert extract_lot_coverage(cov_text) == 50.0
assert extract_adu_coverage_bonus(cov_text) == 10.0
print("All helper function tests passed.")

All helper function tests passed.


In [75]:
def build_table_district_row(section_id: str, district_code: str, district_name: str) -> dict:
    """
    Parse one table-structured district section and return a full row dict.
    Extracts both Standard Option and PRD (Planned Residential Development) values.
    """
    raw = parse_table_district(section_id)

    std_height_text = raw.get("10", {}).get("standard", "")
    prd_height_text = raw.get("10", {}).get("prd", "")
    std_cov_text    = raw.get("11", {}).get("standard", "")
    prd_cov_text    = raw.get("11", {}).get("prd", "")

    height_ft, height_stories             = extract_height(std_height_text)
    prd_height_ft, prd_height_stories     = extract_height(prd_height_text)
    setbacks                               = extract_setbacks(raw.get("5", {}).get("standard", ""))

    row = {
        "district_code":                  district_code,
        "district_name":                  district_name,
        "ordinance_section":              section_id.replace(".B", "").replace("ZO_", "Section "),
        "district_type":                  "residential",
        # ── Standard Option ────────────────────────────────────────────────────
        "max_density_du_per_acre":        extract_ft(raw.get("3", {}).get("standard", "")),
        "max_density_notes":              "",
        "max_height_ft":                  height_ft,
        "max_height_stories":             height_stories,
        "min_lot_width_ft":               extract_ft(raw.get("1", {}).get("standard", "")),
        "min_lot_depth_ft":               extract_ft(raw.get("2", {}).get("standard", "")),
        "max_lot_coverage_pct":           extract_lot_coverage(std_cov_text),
        "adu_coverage_bonus_pct":         extract_adu_coverage_bonus(std_cov_text),
        **setbacks,
        # ── PRD reference values ───────────────────────────────────────────────
        "max_density_du_per_acre_prd":    extract_ft(raw.get("3", {}).get("prd", "")),
        "max_height_ft_prd":              prd_height_ft,
        "max_height_stories_prd":         prd_height_stories,
        "max_lot_coverage_pct_prd":       extract_prd_coverage(prd_cov_text),
        # ── Populated in later sections ────────────────────────────────────────
        "parking_min_per_du":             None,
        "parking_notes":                  "",
        "permits_duplex":                 None,
        "permits_triplex":                None,
        "permits_fourplex":               None,
        "permits_courtyard_apt":          None,
        "permits_cottage_cluster":        None,
        "multifamily_permitted":          None,
        "pud_flag":                       False,
        "wu_code_flag":                   False,
        "dtc_flag":                       False,
        "dtc_character_area":             None,
        "mf_conversion_flag":             False,
        "ri_overlay_eligible":            False,
        "hr_overlay_eligible":            False,
        "notes":                          "",
    }
    return row


# Build all table district rows
table_rows = [
    build_table_district_row(sec_id, code, name)
    for sec_id, code, name in TABLE_DISTRICTS
]

# ── Set overlay eligibility for high-density residential table districts ────────
RI_ELIGIBLE = {"R-3", "R-4", "R-4A", "R-5"}
HR_ELIGIBLE = {"R-4", "R-4A", "R-5"}
for row in table_rows:
    row["ri_overlay_eligible"] = row["district_code"] in RI_ELIGIBLE
    row["hr_overlay_eligible"] = row["district_code"] in HR_ELIGIBLE

# ── Preview ────────────────────────────────────────────────────────────────────
pd.DataFrame(table_rows)[[
    "district_code", "max_density_du_per_acre",
    "max_height_ft", "max_height_stories",
    "max_height_ft_prd", "max_height_stories_prd",
    "max_lot_coverage_pct", "adu_coverage_bonus_pct", "max_lot_coverage_pct_prd",
]]

,district_code,max_density_du_per_acre,max_height_ft,max_height_stories,max_height_ft_prd,max_height_stories_prd,max_lot_coverage_pct,adu_coverage_bonus_pct,max_lot_coverage_pct_prd
0,RE-35,1.2,30.0,2,30.0,2,25.0,None,40.0
1,R1-18,2.0,30.0,2,30.0,2,30.0,None,40.0
2,R1-10,3.5,30.0,2,30.0,3,50.0,None,60.0
3,R1-8,4.5,30.0,2,30.0,3,50.0,None,60.0
4,R1-6,5.5,30.0,2,30.0,3,50.0,None,60.0
5,R-2,10.0,30.0,2,30.0,3,50.0,None,60.0
6,R-3,14.5,30.0,2,30.0,3,50.0,None,60.0
7,R-3A,22.0,30.0,2,40.0,3,50.0,None,60.0
8,R-4,29.0,40.0,3,40.0,3,50.0,None,60.0
9,R-5,43.5,48.0,4,48.0,4,50.0,None,60.0


### ⚠️ Manual review checkpoint — Section 2

Before continuing, verify the table above matches the ordinance source:

- **RE-35**: density 1.2, width 150, height 30 ft / 2 stories, coverage 25%, PRD coverage 40%
- **R1-18**: density 2.0, width 130, height 30 ft / 2 stories, coverage 30%, ADU bonus 10%, PRD coverage 40%
- **R1-10 through R1-6**: height 30 ft / **2 stories** (Standard); **3 stories** PRD — confirm `max_height_stories_prd = 3`
- **R-2, R-3, R-3A**: height 30 ft / 2 stories (Standard); PRD first-150-ft height = 30 ft / 3 stories
- **R-3A PRD**: height 40 ft / 3 stories for first 150 ft — confirm `max_height_ft_prd = 40`
- **R-4**: height 40 ft / 3 stories (Standard)
- **R-5**: height 48 ft / 4 stories (Standard and PRD)
- All R1-10 through R-5 Standard: `adu_coverage_bonus_pct = 10.0`
- `ri_overlay_eligible = True` for R-3, R-4, R-5 (R-4A added in Section 3)
- `hr_overlay_eligible = True` for R-4, R-5 (R-4A added in Section 3)

If any value is wrong, add an override in the next cell rather than editing the parser.

In [76]:
# ── Manual overrides for table district rows ──────────────────────────────────
# Use this cell to correct any misparsed values before continuing.
# Format: next(r for r in table_rows if r["district_code"] == "X")["column"] = value

# Example (uncomment and adjust if needed):
# next(r for r in table_rows if r["district_code"] == "R-3A")["max_height_ft_prd"] = 40.0

# --- Add overrides below this line ---


# --- End overrides ---

# Confirm key assertions
_r110 = next(r for r in table_rows if r["district_code"] == "R1-10")
_r4   = next(r for r in table_rows if r["district_code"] == "R-4")
_r5   = next(r for r in table_rows if r["district_code"] == "R-5")
assert _r110.get("max_height_stories_prd") == 3, f"R1-10 PRD should be 3 stories; got {_r110.get('max_height_stories_prd')}"
assert _r4["max_height_ft"] == 40.0,   f"R-4 standard height should be 40 ft"
assert _r5["max_height_ft"] == 48.0,   f"R-5 standard height should be 48 ft"
print("Table district assertions passed.")

Table district assertions passed.


## Section 3 — Parse prose districts

Districts S-1, RE-43, RE-24, R1-14, and R-4A store their development
standards as prose paragraphs, not structured tables. Each is parsed
individually using regex against the `.B` section text.

After extraction, new schema columns are patched onto each row, and
overlay eligibility flags are set for R-4A.

In [77]:
def get_section_text(section_id: str) -> str:
    """Return the plain text of an ordinance section element."""
    el = soup.find("section", id=section_id)
    if not el:
        raise ValueError(f"{section_id!r} not found")
    return el.get_text(separator=" ", strip=True)


def prose_extract(text: str, pattern: str) -> float | None:
    """Extract a float from prose text using a regex pattern."""
    m = re.search(pattern, text, re.I)
    return float(m.group(1)) if m else None


# ── S-1 (Section 603) ─────────────────────────────────────────────────────────
s1_text = get_section_text("ZO_603.B")

s1_row = {
    "district_code":       "S-1",
    "district_name":       "Suburban Ranch/Farm Residence",
    "ordinance_section":   "Section 603",
    "district_type":       "residential",
    # S-1 minimum lot is 1 acre; density is effectively 1 DU/AC
    "max_density_du_per_acre": 1.0,
    "max_density_notes":   "1-acre minimum net lot area; density ~1 DU/acre",
    "max_height_ft":       prose_extract(s1_text, r'(\d+)\s*feet'),
    "max_height_stories":  prose_extract(s1_text, r'(\d+)\s*stor'),
    "min_lot_width_ft":    None,   # S-1 has no minimum width
    "min_lot_depth_ft":    None,   # S-1 has no minimum depth
    "max_lot_coverage_pct": 20.0,  # 20% for lots ≤2 acres
    "min_front_setback_ft":           prose_extract(s1_text, r'front\s+setback\s+is\s+(\d+)'),
    "min_rear_setback_ft":            prose_extract(s1_text, r'rear\s+setback\s+is\s+(\d+)'),
    "min_street_side_setback_ft":     None,
    "min_interior_side_setback_a_ft": prose_extract(s1_text, r'side\s+setback\s+is\s+(\d+)'),
    "min_interior_side_setback_b_ft": prose_extract(s1_text, r'side\s+setback\s+is\s+(\d+)'),
    "max_density_du_per_acre_prd":    None,
    "max_height_ft_prd":              None,
    "parking_min_per_du":             None,
    "parking_notes":                  "",
    "permits_duplex":                 None,
    "permits_triplex":                None,
    "permits_fourplex":               None,
    "permits_courtyard_apt":          None,
    "permits_cottage_cluster":        None,
    "multifamily_permitted":          None,
    "pud_flag":                       False,
    "wu_code_flag":                   False,
    "notes":                          "1 primary DU + up to 2 ADUs per lot; lot coverage 20% for ≤2 acres, 10% for >2 acres",
}

print("S-1 extracted:")
for k, v in s1_row.items():
    if v is not None and v != "" and v is not False:
        print(f"  {k}: {v}")

S-1 extracted:
  district_code: S-1
  district_name: Suburban Ranch/Farm Residence
  ordinance_section: Section 603
  district_type: residential
  max_density_du_per_acre: 1.0
  max_density_notes: 1-acre minimum net lot area; density ~1 DU/acre
  max_height_ft: 40.0
  max_lot_coverage_pct: 20.0
  min_front_setback_ft: 40.0
  min_rear_setback_ft: 30.0
  min_interior_side_setback_a_ft: 30.0
  min_interior_side_setback_b_ft: 30.0
  notes: 1 primary DU + up to 2 ADUs per lot; lot coverage 20% for ≤2 acres, 10% for >2 acres


In [78]:
# ── RE-43 (Section 605) ───────────────────────────────────────────────────────
re43_text = get_section_text("ZO_605.B")

re43_row = {
    "district_code":       "RE-43",
    "district_name":       "Residential Estate — 43,560 sqft minimum",
    "ordinance_section":   "Section 605",
    "district_type":       "residential",
    "max_density_du_per_acre": 1.0,
    "max_density_notes":   "43,560 sqft (1 acre) minimum lot; effectively 1 DU/acre",
    "max_height_ft":       30.0,
    "max_height_stories":  2,
    "min_lot_width_ft":    prose_extract(re43_text, r'minimum width of (\d+)'),
    "min_lot_depth_ft":    prose_extract(re43_text, r'minimum depth of (\d+)'),
    "max_lot_coverage_pct": 20.0,
    "min_front_setback_ft":           prose_extract(re43_text, r'front\s+setback\s+is\s+(\d+)'),
    "min_rear_setback_ft":            prose_extract(re43_text, r'rear\s+setback\s+is\s+(\d+)'),
    "min_street_side_setback_ft":     None,
    "min_interior_side_setback_a_ft": prose_extract(re43_text, r'side\s+setback\s+is\s+(\d+)'),
    "min_interior_side_setback_b_ft": prose_extract(re43_text, r'side\s+setback\s+is\s+(\d+)'),
    "max_density_du_per_acre_prd":    None,
    "max_height_ft_prd":              None,
    "parking_min_per_du":             None,
    "parking_notes":                  "",
    "permits_duplex":                 None,
    "permits_triplex":                None,
    "permits_fourplex":               None,
    "permits_courtyard_apt":          None,
    "permits_cottage_cluster":        None,
    "multifamily_permitted":          None,
    "pud_flag":                       False,
    "wu_code_flag":                   False,
    "notes":                          "Applies only to land zoned RE-43 prior to September 13, 1981",
}

# ── RE-24 (Section 606) ───────────────────────────────────────────────────────
re24_text = get_section_text("ZO_606.B")

re24_row = {
    "district_code":       "RE-24",
    "district_name":       "Residential Estate — 24,000 sqft minimum",
    "ordinance_section":   "Section 606",
    "district_type":       "residential",
    "max_density_du_per_acre": 1.8,
    "max_density_notes":   "24,000 sqft minimum lot; effectively ~1.8 DU/acre",
    "max_height_ft":       30.0,
    "max_height_stories":  2,
    "min_lot_width_ft":    prose_extract(re24_text, r'minimum width of (\d+)'),
    "min_lot_depth_ft":    prose_extract(re24_text, r'minimum depth of (\d+)'),
    "max_lot_coverage_pct": 25.0,
    "min_front_setback_ft":           prose_extract(re24_text, r'front\s+setback\s+is\s+(\d+)'),
    "min_rear_setback_ft":            prose_extract(re24_text, r'rear\s+setback\s+is\s+(\d+)'),
    "min_street_side_setback_ft":     prose_extract(re24_text, r'street side\s+setback\s+is\s+(\d+)'),
    "min_interior_side_setback_a_ft": prose_extract(re24_text, r'interior side\s+setback\s+is\s+(\d+)'),
    "min_interior_side_setback_b_ft": prose_extract(re24_text, r'interior side\s+setback\s+is\s+(\d+)'),
    "max_density_du_per_acre_prd":    None,
    "max_height_ft_prd":              None,
    "parking_min_per_du":             None,
    "parking_notes":                  "",
    "permits_duplex":                 None,
    "permits_triplex":                None,
    "permits_fourplex":               None,
    "permits_courtyard_apt":          None,
    "permits_cottage_cluster":        None,
    "multifamily_permitted":          None,
    "pud_flag":                       False,
    "wu_code_flag":                   False,
    "notes":                          "Applies only to land zoned prior to September 13, 1981",
}

# ── R1-14 (Section 607) ───────────────────────────────────────────────────────
r114_text = get_section_text("ZO_607.B")

r114_row = {
    "district_code":       "R1-14",
    "district_name":       "Single-Family Residence — 14,000 sqft minimum",
    "ordinance_section":   "Section 607",
    "district_type":       "residential",
    "max_density_du_per_acre": 3.1,
    "max_density_notes":   "14,000 sqft minimum lot; effectively ~3.1 DU/acre",
    "max_height_ft":       30.0,
    "max_height_stories":  2,
    "min_lot_width_ft":    prose_extract(r114_text, r'minimum width of (\d+)'),
    "min_lot_depth_ft":    prose_extract(r114_text, r'minimum depth of (\d+)'),
    "max_lot_coverage_pct": 25.0,
    "min_front_setback_ft":           prose_extract(r114_text, r'front\s+setback\s+is\s+(\d+)'),
    "min_rear_setback_ft":            prose_extract(r114_text, r'rear\s+setback\s+is\s+(\d+)'),
    "min_street_side_setback_ft":     prose_extract(r114_text, r'street side\s+setback\s+is\s+(\d+)'),
    "min_interior_side_setback_a_ft": prose_extract(r114_text, r'interior side\s+setback\s+is\s+(\d+)'),
    "min_interior_side_setback_b_ft": prose_extract(r114_text, r'interior side\s+setback\s+is\s+(\d+)'),
    "max_density_du_per_acre_prd":    None,
    "max_height_ft_prd":              None,
    "parking_min_per_du":             None,
    "parking_notes":                  "",
    "permits_duplex":                 None,
    "permits_triplex":                None,
    "permits_fourplex":               None,
    "permits_courtyard_apt":          None,
    "permits_cottage_cluster":        None,
    "multifamily_permitted":          None,
    "pud_flag":                       False,
    "wu_code_flag":                   False,
    "notes":                          "Applies only to land zoned R1-14 prior to September 13, 1981",
}

# ── R-4A (Section 619) ────────────────────────────────────────────────────────
# R-4A expresses density as sqft per unit, not DU/acre
# 1,000 sqft/DU → 43,560 / 1,000 = 43.56 DU/acre theoretical max
r4a_text = get_section_text("ZO_619.B")

r4a_row = {
    "district_code":       "R-4A",
    "district_name":       "Multi-Family Residence General",
    "ordinance_section":   "Section 619",
    "district_type":       "residential",
    "max_density_du_per_acre": 43.56,
    "max_density_notes":   "1,000 sqft lot area per DU (500 sqft for efficiency); min lot 6,000 sqft, min width 60 ft",
    "max_height_ft":       48.0,
    "max_height_stories":  None,
    "min_lot_width_ft":    60.0,
    "min_lot_depth_ft":    94.0,
    "max_lot_coverage_pct": 50.0,
    "min_front_setback_ft":           prose_extract(r4a_text, r'front\s+yard.*?(\d+)\s*feet'),
    "min_rear_setback_ft":            prose_extract(r4a_text, r'rear\s+yard.*?(\d+)\s*feet'),
    "min_street_side_setback_ft":     None,
    "min_interior_side_setback_a_ft": prose_extract(r4a_text, r'side\s+yards.*?(\d+)\s*feet'),
    "min_interior_side_setback_b_ft": prose_extract(r4a_text, r'side\s+yards.*?(\d+)\s*feet'),
    "max_density_du_per_acre_prd":    None,
    "max_height_ft_prd":              None,
    "parking_min_per_du":             None,
    "parking_notes":                  "",
    "permits_duplex":                 None,
    "permits_triplex":                None,
    "permits_fourplex":               None,
    "permits_courtyard_apt":          None,
    "permits_cottage_cluster":        None,
    "multifamily_permitted":          None,
    "pud_flag":                       False,
    "wu_code_flag":                   False,
    "notes":                          "Density expressed as sqft/DU; 1,000 sqft per DU, 500 sqft per efficiency apartment",
}

prose_rows = [s1_row, re43_row, re24_row, r114_row, r4a_row]
print(f"Built {len(prose_rows)} prose district rows.")

Built 5 prose district rows.


In [79]:
# ── Patch new schema columns onto prose rows (not present in their row dicts) ──
_NEW_COLS = {
    "adu_coverage_bonus_pct":  None,
    "max_height_stories_prd":  None,
    "max_lot_coverage_pct_prd": None,
    "dtc_flag":                False,
    "dtc_character_area":      None,
    "mf_conversion_flag":      False,
    "ri_overlay_eligible":     False,
    "hr_overlay_eligible":     False,
    "mh_overlay_eligible":     False,
    "hp_overlay_eligible":     False,
}
for row in prose_rows:
    for col, default in _NEW_COLS.items():
        row.setdefault(col, default)

# R-4A: overlay eligibility (§630 R-I eligible; §631 H-R eligible)
_r4a = next(r for r in prose_rows if r["district_code"] == "R-4A")
_r4a["ri_overlay_eligible"] = True
_r4a["hr_overlay_eligible"] = True

# RE-35: special coverage — 25% standard, 40% if all structures <20 ft/1-story
# adu_coverage_bonus_pct stays None (no ADU coverage bonus for RE-35)

print(f"Patched {len(prose_rows)} prose district rows with new schema columns.")
print("R-4A ri_overlay_eligible:", _r4a["ri_overlay_eligible"])
print("R-4A hr_overlay_eligible:", _r4a["hr_overlay_eligible"])

Patched 5 prose district rows with new schema columns.
R-4A ri_overlay_eligible: True
R-4A hr_overlay_eligible: True


### ⚠️ Manual review checkpoint — prose districts

Verify extracted prose district values and add overrides below if needed:

| District | Density | Width | Height | Coverage |
|---|---|---|---|---|
| S-1 | 1.0 DU/acre | None | ~30 ft / 2 stories | 20% (≤2 acres) |
| RE-43 | 1.0 | — | 30 ft / 2 | 20% |
| RE-24 | 1.8 | — | 30 ft / 2 | 25% |
| R1-14 | 3.1 | — | 30 ft / 2 | 25% |
| R-4A | 43.56 | 60 ft | 48 ft | 50% |

R-4A: `ri_overlay_eligible = True`, `hr_overlay_eligible = True`.

In [80]:
# ── Manual verification — EDIT THIS CELL if regex misparsed anything ──────────
#
# Compare the extracted values below against the ordinance source.
# If a value is wrong, override it here rather than editing the regex above,
# so that the correction is visible and auditable.
#
# Format:  rows[i]["column_name"] = correct_value
# Example: prose_rows[0]["min_front_setback_ft"] = 40.0  # S-1 front setback per 603.B.2.a

# --- Add overrides below this line ---


# --- End overrides ---

pd.DataFrame(prose_rows)[[
    "district_code", "max_density_du_per_acre", "min_lot_width_ft",
    "max_height_ft", "min_front_setback_ft", "max_lot_coverage_pct"
]]

,district_code,max_density_du_per_acre,min_lot_width_ft,max_height_ft,min_front_setback_ft,max_lot_coverage_pct
0,S-1,1.00,NaN,40.0,40.0,20.0
1,RE-43,1.00,165.0,30.0,40.0,20.0
2,RE-24,1.80,130.0,30.0,30.0,25.0
3,R1-14,3.10,110.0,30.0,30.0,25.0
4,R-4A,43.56,60.0,48.0,20.0,50.0


## Section 3b — Commercial and special residential districts

Adds rows for R-O, C-O, C-1, C-2, C-3, and UR. These districts either
permit residential uses directly or are explicit hybrid residential-commercial zones.

**Key regulatory basis (from §§620–624, 642):**
- **C-1 / C-2 / C-3**: Residential uses governed by R-3 standards (14.5 DU/acre, 30 ft/2 stories).
  Core areas and Central Ave (Camelback–Harrison): City Council may permit up to 56 ft/4 stories.
  City Council may also allow up to R-5 standards in village cores.
- **R-O**: Residential-office transition; permits residential continuation or reconversion.
  No numeric development standards in the ordinance text — references "residential scale." ⚠️ Manual review required.
- **C-O**: Residential as accessory use only. H-R overlay eligible.
- **UR**: Min 40 DU/gross acre; no max; max 75 ft; no lot coverage maximum.
  Bounded geography: 7th Ave to 7th St, Lincoln St to Grand Canal.

All six districts are eligible for the §711 multi-family conversion pathway
(except R-O and C-O, which are classified separately).

In [81]:
# ── Shared base for C-1/C-2/C-3 commercial districts ──────────────────────────
# Residential uses governed by R-3 standards per §622/623/624.E.1.
# max_height_ft = 30 ft is the non-core baseline; core areas allow 56 ft.
# Document core exception in notes; flag core-area parcels via spatial join in scanner.

_C_COMMERCIAL_BASE = {
    "district_type":                  "commercial",
    "max_density_du_per_acre":        14.5,       # R-3 standard (§615)
    "max_density_notes":              (
        "Residential governed by R-3 standards (§622/623/624.E.1). "
        "Core areas + Central Ave (Camelback–Harrison): max 4 stories/56 ft per City Council. "
        "City Council may permit up to R-5 standards (43.5 DU/acre) in village core areas."
    ),
    "max_height_ft":                  30.0,        # Non-core/non-Central-Ave baseline
    "max_height_stories":             2,
    "min_lot_width_ft":               60.0,        # R-3 (§615)
    "min_lot_depth_ft":               94.0,
    "max_lot_coverage_pct":           50.0,        # R-3 standard
    "adu_coverage_bonus_pct":         10.0,        # R-3 inherits ADU/shade bonus
    "min_front_setback_ft":           25.0,        # R-3 standard
    "min_rear_setback_ft":            15.0,
    "min_street_side_setback_ft":     10.0,
    "min_interior_side_setback_a_ft": 10.0,
    "min_interior_side_setback_b_ft": 3.0,         # R-3 asymmetric interior side
    "max_density_du_per_acre_prd":    15.5,        # R-3 PRD
    "max_height_ft_prd":              30.0,
    "max_height_stories_prd":         3,
    "max_lot_coverage_pct_prd":       60.0,
    "parking_min_per_du":             None,        # Populated in Section 5
    "parking_notes":                  "",
    "permits_duplex":                 True,
    "permits_triplex":                True,
    "permits_fourplex":               True,
    "permits_courtyard_apt":          True,
    "permits_cottage_cluster":        True,
    "multifamily_permitted":          True,
    "pud_flag":                       False,
    "wu_code_flag":                   False,
    "dtc_flag":                       False,
    "dtc_character_area":             None,
    "mf_conversion_flag":             True,        # §711 commercial building conversion eligible
    "ri_overlay_eligible":            False,
    "hr_overlay_eligible":            True,        # §631 H-R overlay eligible
}

c1_row = {
    **_C_COMMERCIAL_BASE,
    "district_code":    "C-1",
    "district_name":    "Commercial Neighborhood Retail",
    "ordinance_section": "Section 622",
    "notes": (
        "Residential governed by R-3 standards (§622.E.1). "
        "Core areas + Central Ave: max 4 stories/56 ft (City Council approval required). "
        "MH Overlay eligible when platted as SFI subdivision (§632.C.2.c). "
        "SFI subdivisions permitted per §608.I. "
        "Wholesaling prohibited; all uses in closed buildings. "
        "§711 multi-family conversion eligible (commercial building must be obsolete)."
    ),
}

c2_row = {
    **_C_COMMERCIAL_BASE,
    "district_code":    "C-2",
    "district_name":    "Commercial Intermediate",
    "ordinance_section": "Section 623",
    "notes": (
        "Residential governed by R-3 standards (§623.E.1). "
        "Core areas + Central Ave: max 4 stories/56 ft. "
        "Canal ROW setbacks: avg 20 ft (≤2 stories), avg 30 ft (>2 stories). "
        "MH Overlay eligible when platted as SFI subdivision. "
        "§711 multi-family conversion eligible."
    ),
}

c3_row = {
    **_C_COMMERCIAL_BASE,
    "district_code":    "C-3",
    "district_name":    "General Commercial",
    "ordinance_section": "Section 624",
    "notes": (
        "Residential governed by R-3 standards (§624.E.1). "
        "Core areas + Central Ave: max 4 stories/56 ft. "
        "30-ft building setback required from RE-43 through R1-6 and PAD districts. "
        "MH Overlay eligible when platted as SFI subdivision. "
        "§711 multi-family conversion eligible."
    ),
}

# ── R-O (Section 620) — Residential Office ────────────────────────────────────
ro_row = {
    "district_code":                  "R-O",
    "district_name":                  "Residential Office — Restricted Commercial",
    "ordinance_section":              "Section 620",
    "district_type":                  "residential_commercial",
    "max_density_du_per_acre":        None,
    "max_density_notes":              (
        "Residential continuation or reconversion permitted (§620). "
        "No explicit density limit stated — development at residential scale. "
        "⚠️ Manual review required; contact City of Phoenix PDD for numeric standards."
    ),
    "max_height_ft":                  None,
    "max_height_stories":             None,
    "min_lot_width_ft":               None,
    "min_lot_depth_ft":               None,
    "max_lot_coverage_pct":           None,
    "adu_coverage_bonus_pct":         None,
    "min_front_setback_ft":           None,
    "min_rear_setback_ft":            None,
    "min_street_side_setback_ft":     None,
    "min_interior_side_setback_a_ft": None,
    "min_interior_side_setback_b_ft": None,
    "max_density_du_per_acre_prd":    None,
    "max_height_ft_prd":              None,
    "max_height_stories_prd":         None,
    "max_lot_coverage_pct_prd":       None,
    "parking_min_per_du":             2.0,
    "parking_notes":                  "Same as single-family detached: 2 spaces/DU (§702.C)",
    "permits_duplex":                 False,
    "permits_triplex":                False,
    "permits_fourplex":               False,
    "permits_courtyard_apt":          False,
    "permits_cottage_cluster":        False,
    "multifamily_permitted":          False,
    "pud_flag":                       False,
    "wu_code_flag":                   False,
    "dtc_flag":                       False,
    "dtc_character_area":             None,
    "mf_conversion_flag":             False,
    "ri_overlay_eligible":            False,
    "hr_overlay_eligible":            True,    # §631 H-R overlay eligible
    "notes": (
        "Transition district at arterial edges of residential areas (§620). "
        "Permits new development at residential scale, office conversion, "
        "or continuation/reconversion of residential uses. "
        "⚠️ No numeric setback/height/coverage standards tabulated in the ordinance. "
        "All numeric fields are NULL — manual override required from city development records."
    ),
}

# ── C-O (Section 621) — Commercial Office ────────────────────────────────────
co_row = {
    "district_code":                  "C-O",
    "district_name":                  "Commercial Office — Restricted Commercial",
    "ordinance_section":              "Section 621",
    "district_type":                  "commercial",
    "max_density_du_per_acre":        None,
    "max_density_notes":              (
        "Residential permitted as accessory use only — not a primary use (§621). "
        "No standalone residential density applicable."
    ),
    "max_height_ft":                  None,
    "max_height_stories":             None,
    "min_lot_width_ft":               None,
    "min_lot_depth_ft":               None,
    "max_lot_coverage_pct":           None,
    "adu_coverage_bonus_pct":         None,
    "min_front_setback_ft":           None,
    "min_rear_setback_ft":            None,
    "min_street_side_setback_ft":     None,
    "min_interior_side_setback_a_ft": None,
    "min_interior_side_setback_b_ft": None,
    "max_density_du_per_acre_prd":    None,
    "max_height_ft_prd":              None,
    "max_height_stories_prd":         None,
    "max_lot_coverage_pct_prd":       None,
    "parking_min_per_du":             None,
    "parking_notes":                  "Accessory residential: multi-family rate applies (1.5 spaces/DU per §702.C)",
    "permits_duplex":                 False,
    "permits_triplex":                False,
    "permits_fourplex":               False,
    "permits_courtyard_apt":          False,
    "permits_cottage_cluster":        False,
    "multifamily_permitted":          False,
    "pud_flag":                       False,
    "wu_code_flag":                   False,
    "dtc_flag":                       False,
    "dtc_character_area":             None,
    "mf_conversion_flag":             True,        # Office buildings eligible for §711
    "ri_overlay_eligible":            False,
    "hr_overlay_eligible":            True,        # §631
    "notes": (
        "Commercial office transition district (§621). "
        "General Office and Major Office options (pre/post-1986 regulations differ). "
        "Residential permitted as accessory use only — not a primary permitted use. "
        "H-R overlay eligible (§631). "
        "§711 multi-family conversion eligible for existing office buildings."
    ),
}

# ── UR (Section 642) — Urban Residential ──────────────────────────────────────
ur_row = {
    "district_code":                  "UR",
    "district_name":                  "Urban Residential",
    "ordinance_section":              "Section 642",
    "district_type":                  "residential",
    "max_density_du_per_acre":        None,
    "max_density_notes":              (
        "Minimum 40 DU/gross acre required; no maximum density (§642.E). "
        "Bounded geography: 7th Ave–7th St, Lincoln St–Grand Canal. "
        "Attached multi-family only; recreational amenities required on-site."
    ),
    "max_height_ft":                  75.0,
    "max_height_stories":             None,
    "min_lot_width_ft":               None,
    "min_lot_depth_ft":               None,
    "max_lot_coverage_pct":           None,        # No maximum (§642.G)
    "adu_coverage_bonus_pct":         None,
    "min_front_setback_ft":           0.0,         # No minimum; 10-ft max for ≥65% of frontage
    "min_rear_setback_ft":            None,        # Governed by interior setback table (§642.F)
    "min_street_side_setback_ft":     None,
    "min_interior_side_setback_a_ft": None,        # 1 ft per 1 ft height (0–75 ft) vs low-density
    "min_interior_side_setback_b_ft": None,
    "max_density_du_per_acre_prd":    None,
    "max_height_ft_prd":              None,
    "max_height_stories_prd":         None,
    "max_lot_coverage_pct_prd":       None,
    "parking_min_per_du":             1.0,         # 1 space/1BR; 1.5 spaces/2+BR (§642.I.1)
    "parking_notes":                  (
        "1 space/1BR or smaller unit; 1.5 spaces/2+BR unit (§642.I.1). "
        "All residential parking in structures/enclosed garages, not visible from ROW. "
        "Nonresidential accessory: 1 space/300 sqft. Shared parking option available."
    ),
    "permits_duplex":                 True,
    "permits_triplex":                True,
    "permits_fourplex":               True,
    "permits_courtyard_apt":          True,
    "permits_cottage_cluster":        True,
    "multifamily_permitted":          True,
    "pud_flag":                       False,
    "wu_code_flag":                   False,
    "dtc_flag":                       False,
    "dtc_character_area":             None,
    "mf_conversion_flag":             False,
    "ri_overlay_eligible":            False,
    "hr_overlay_eligible":            False,
    "notes": (
        "High-density pedestrian-oriented multi-family (§642). "
        "Min 40 DU/gross acre; no maximum density. Max 75 ft; no lot coverage max. "
        "Street setback: 10-ft maximum for ≥65% of building frontage; no minimum for remainder. "
        "Interior setbacks: 1 ft per 1 ft height (abutting S-1/RE-43 through R-2/PADs 1-11); "
        "10 ft (abutting commercial and higher-density districts). Max 75-ft setback. "
        "75% of sidewalk must be shaded (§642.H.1). "
        "Ground floor neighborhood commercial permitted as accessory use (≤5,000 sqft/business). "
        "⚠️ min_front_setback_ft = 0.0 reflects no minimum; 10-ft MAXIMUM applies to ≥65% of frontage."
    ),
}

commercial_rows = [ro_row, co_row, c1_row, c2_row, c3_row, ur_row]
print(f"Built {len(commercial_rows)} commercial/special district rows:")
pd.DataFrame(commercial_rows)[[
    "district_code", "district_type", "max_density_du_per_acre",
    "max_height_ft", "multifamily_permitted", "mf_conversion_flag", "hr_overlay_eligible",
]]

Built 6 commercial/special district rows:


,district_code,district_type,max_density_du_per_acre,max_height_ft,multifamily_permitted,mf_conversion_flag,hr_overlay_eligible
0,R-O,residential_commercial,NaN,NaN,False,False,True
1,C-O,commercial,NaN,NaN,False,True,True
2,C-1,commercial,14.5,30.0,True,True,True
3,C-2,commercial,14.5,30.0,True,True,True
4,C-3,commercial,14.5,30.0,True,True,True
5,UR,residential,NaN,75.0,True,False,False


### ⚠️ Manual review checkpoint — commercial districts

- **R-O**: All numeric standards are NULL. If city development records provide setback/height
  data for R-O, override below.
- **C-1/C-2/C-3**: `max_height_ft = 30` is the non-core baseline. Core area parcels
  (within village cores + Central Ave Camelback–Harrison) allow 56 ft/4 stories via City Council.
  These will be flagged in the scanner via a spatial join to the General Plan Village Core GIS layer.
- **UR**: `min_front_setback_ft = 0.0` is correct — represents no minimum (10-ft max applies to ≥65% of frontage).
- **UR**: `max_density_du_per_acre = NULL` is correct — minimum is 40 DU/acre, there is no maximum.

In [82]:
# ── Manual overrides for commercial district rows ─────────────────────────────
# Format: ro_row["column"] = value

# R-O: If standards are found, override here:
# ro_row["max_height_ft"]               = X
# ro_row["max_height_stories"]          = Y
# ro_row["min_front_setback_ft"]        = Z
# ro_row["min_interior_side_setback_a_ft"] = W

# --- Add overrides below this line ---


# --- End overrides ---

assert c1_row["max_density_du_per_acre"] == 14.5
assert c1_row["max_height_ft"] == 30.0
assert ur_row["max_height_ft"] == 75.0
assert ur_row["max_density_du_per_acre"] is None
print("Commercial district assertions passed.")

Commercial district assertions passed.


## Section 3c — Overlay districts: R-I and H-R

R-I (Residential Infill) and H-R (High-Rise) are overlay districts that modify
underlying base district standards. They are stored as anchor rows for reference,
not full development standard rows. The `ri_overlay_eligible` and `hr_overlay_eligible`
flags on base district rows already track applicability.

**R-I (§630):** Bounded geography — 19th Ave, 24th St, Thomas Rd, Harrison St.
Eligible base zones: R-3, R-4, R-4A, R-5. Increases density beyond base limits.

**H-R (§631):** No geographic boundary — applies citywide where combined with eligible
base zones. Eligible: R-4, R-4A, R-5, C-O, C-1, C-2, C-3, A-1, A-2, P-1, P-2.
For development >4 stories/56 ft: 450 sqft/DU; max 250 ft.

In [83]:
# ── R-I Residential Infill Overlay (Section 630) ──────────────────────────────
ri_row = {
    "district_code":                  "R-I",
    "district_name":                  "Residential Infill Overlay",
    "ordinance_section":              "Section 630",
    "district_type":                  "overlay",
    "max_density_du_per_acre":        None,
    "max_density_notes":              (
        "Combined R-3: 2,000 sqft/DU (≈21.8 DU/acre). "
        "Combined R-4: 1,250 sqft/DU (≈34.8 DU/acre). "
        "Boundary: 19th Ave / 24th St / Thomas Rd / Harrison (Railroad)."
    ),
    "max_height_ft":                  65.0,
    "max_height_stories":             6,
    "min_lot_width_ft":               60.0,
    "min_lot_depth_ft":               94.0,
    "max_lot_coverage_pct":           None,
    "adu_coverage_bonus_pct":         None,
    "min_front_setback_ft":           None,
    "min_rear_setback_ft":            None,
    "min_street_side_setback_ft":     None,
    "min_interior_side_setback_a_ft": None,
    "min_interior_side_setback_b_ft": None,
    "max_density_du_per_acre_prd":    None,
    "max_height_ft_prd":              None,
    "max_height_stories_prd":         None,
    "max_lot_coverage_pct_prd":       None,
    "parking_min_per_du":             None,
    "parking_notes":                  "Per underlying base district",
    "permits_duplex":                 True,
    "permits_triplex":                True,
    "permits_fourplex":               True,
    "permits_courtyard_apt":          True,
    "permits_cottage_cluster":        True,
    "multifamily_permitted":          True,
    "pud_flag":                       False,
    "wu_code_flag":                   False,
    "dtc_flag":                       False,
    "dtc_character_area":             None,
    "mf_conversion_flag":             False,
    "ri_overlay_eligible":            True,
    "hr_overlay_eligible":            False,
    "notes": (
        "Overlay increases density within bounded central Phoenix geography (§630). "
        "Eligible base zones: R-3, R-4, R-4A, R-5. "
        "Min lot: 6,000 sqft; min width: 60 ft; min depth: 94 ft. "
        "Setbacks/coverage: per underlying base district. "
        "Combined with R-4A/R-5: max 6 stories/65 ft. "
        "R-5 base with commercial: ≥50% gross floor area must be residential. "
        "Development review (§507) required."
    ),
}

# ── H-R High-Rise Overlay (Section 631) ──────────────────────────────────────
hr_row = {
    "district_code":                  "H-R",
    "district_name":                  "High-Rise Overlay",
    "ordinance_section":              "Section 631",
    "district_type":                  "overlay",
    "max_density_du_per_acre":        96.8,
    "max_density_notes":              (
        "For development >4 stories/56 ft: 450 sqft/DU (≈96.8 DU/acre), "
        "300 sqft/efficiency, 200 sqft/guestroom. "
        "≤4 stories/56 ft: base district standards apply. "
        "City Council may permit >250 ft (§631.B.2.f(1)); airport limits apply."
    ),
    "max_height_ft":                  250.0,
    "max_height_stories":             None,
    "min_lot_width_ft":               None,
    "min_lot_depth_ft":               None,
    "max_lot_coverage_pct":           50.0,
    "adu_coverage_bonus_pct":         None,
    "min_front_setback_ft":           None,
    "min_rear_setback_ft":            None,
    "min_street_side_setback_ft":     None,
    "min_interior_side_setback_a_ft": None,
    "min_interior_side_setback_b_ft": None,
    "max_density_du_per_acre_prd":    None,
    "max_height_ft_prd":              None,
    "max_height_stories_prd":         None,
    "max_lot_coverage_pct_prd":       None,
    "parking_min_per_du":             None,
    "parking_notes":                  "Per base district for ≤4 stories/56 ft",
    "permits_duplex":                 True,
    "permits_triplex":                True,
    "permits_fourplex":               True,
    "permits_courtyard_apt":          True,
    "permits_cottage_cluster":        True,
    "multifamily_permitted":          True,
    "pud_flag":                       False,
    "wu_code_flag":                   False,
    "dtc_flag":                       False,
    "dtc_character_area":             None,
    "mf_conversion_flag":             False,
    "ri_overlay_eligible":            False,
    "hr_overlay_eligible":            True,
    "notes": (
        "High-rise overlay increases height and density (§631). "
        "Eligible base zones: R-4, R-4A, R-5, C-O, C-1, C-2, C-3, A-1, A-2, P-1, P-2. "
        "≤4 stories/56 ft: base district standards. "
        ">4 stories: 450 sqft/DU min area per unit; 50% max lot coverage. "
        "Max 250 ft; City Council can exceed (no absolute upper limit). "
        "Sky Harbor Airport height limits apply and cannot be varied. "
        "Development review (§507) required."
    ),
}

overlay_rows = [ri_row, hr_row]
print(f"Built {len(overlay_rows)} overlay rows: {[r['district_code'] for r in overlay_rows]}")

Built 2 overlay rows: ['R-I', 'H-R']


## Section 3d — Downtown Code character areas (Chapter 12)

The Downtown Code (DTC) applies within: McDowell Rd (N) / 7th St (E) / Buckeye Rd (S) / 7th Ave (W).
15 character areas regulate use, height, and density.

**Height and density** are governed by regulating maps (§1202.B–C) — not text.
All `max_height_ft` and `max_density_du_per_acre` fields are NULL.
Populate from the City of Phoenix DTC GIS layer.

**Land use** from Table 1203.C:
- Multi-Family (DU, Multi-Family): **P** in all 15 character areas.
- SFD Detached: **NP** in BioMed, Business Core, Downtown Gateway, Warehouse.
- SFA Attached: **NP** in BioMed, Business Core, Downtown Gateway, Warehouse.

DTC overrides the general Zoning Ordinance where conflicts exist (§1201.B.2).
HP-zoned properties additionally subject to Chapter 8 (HP prevails over DTC).

In [84]:
# ── Downtown Code character area rows ─────────────────────────────────────────
# (code, area_name, sfd_permitted, sfa_permitted, ordinance_section)
_DTC_AREAS = [
    ("DTC-BM",  "Biomed",               False, False, "Section 1208"),
    ("DTC-BC",  "Business Core",        False, False, "Section 1209"),
    ("DTC-CP",  "Central Park",         True,  True,  "Section 1210"),
    ("DTC-CC",  "Commercial Corridors", True,  True,  "Section 1211"),
    ("DTC-DG",  "Downtown Gateway",     False, False, "Section 1212"),
    ("DTC-EE",  "East Evergreen",       True,  True,  "Section 1213"),
    ("DTC-ECE", "Evans Churchill East", True,  True,  "Section 1214"),
    ("DTC-ECW", "Evans Churchill West", True,  True,  "Section 1215"),
    ("DTC-MC",  "McDowell Corridor",    True,  True,  "Section 1216"),
    ("DTC-RE",  "Roosevelt East",       True,  True,  "Section 1217"),
    ("DTC-RN",  "Roosevelt North",      True,  True,  "Section 1218"),
    ("DTC-RS",  "Roosevelt South",      True,  True,  "Section 1219"),
    ("DTC-TP",  "Townsend Park",        True,  True,  "Section 1220"),
    ("DTC-VB",  "Van Buren",            True,  True,  "Section 1221"),
    ("DTC-WH",  "Warehouse",            False, True,  "Section 1222"),  # SFD NP; SFA P
]

_DTC_BASE = {
    "district_type":                  "form_based_code",
    "max_density_du_per_acre":        None,
    "max_density_notes":              "Governed by §1202.C density map (image). Bonus per §1223 sustainability points.",
    "max_height_ft":                  None,
    "max_height_stories":             None,
    "min_lot_width_ft":               None,
    "min_lot_depth_ft":               None,
    "max_lot_coverage_pct":           None,
    "adu_coverage_bonus_pct":         None,
    "min_front_setback_ft":           None,
    "min_rear_setback_ft":            None,
    "min_street_side_setback_ft":     None,
    "min_interior_side_setback_a_ft": None,
    "min_interior_side_setback_b_ft": None,
    "max_density_du_per_acre_prd":    None,
    "max_height_ft_prd":              None,
    "max_height_stories_prd":         None,
    "max_lot_coverage_pct_prd":       None,
    "parking_min_per_du":             None,
    "parking_notes":                  "Residential: per §1206. Nonresidential: none required in most areas.",
    "permits_duplex":                 True,
    "permits_triplex":                True,
    "permits_fourplex":               True,
    "permits_courtyard_apt":          True,
    "permits_cottage_cluster":        True,
    "multifamily_permitted":          True,
    "pud_flag":                       False,
    "wu_code_flag":                   False,
    "dtc_flag":                       True,
    "mf_conversion_flag":             True,    # Commercial buildings in DTC eligible for §711
    "ri_overlay_eligible":            False,
    "hr_overlay_eligible":            False,
}

dtc_rows = []
for code, area_name, sfd_ok, sfa_ok, section in _DTC_AREAS:
    dtc_rows.append({
        **_DTC_BASE,
        "district_code":      code,
        "district_name":      f"Downtown Code — {area_name}",
        "ordinance_section":  section,
        "dtc_character_area": area_name,
        "notes": (
            f"DTC character area: {area_name} ({section}). "
            f"SFD detached: {'permitted' if sfd_ok else 'NOT PERMITTED'}. "
            f"SFA attached: {'permitted' if sfa_ok else 'NOT PERMITTED'}. "
            "MF: permitted by right (Table 1203.C). "
            "Height/density from §1202.B–C regulating maps (not text — populate from GIS). "
            "DTC overrides general ZO where conflicts occur (§1201.B.2). "
            "HP-zoned properties subject to Chapter 8 (HP prevails over DTC). "
            "§711 multi-family conversion eligible for qualifying commercial buildings."
        ),
    })

print(f"Built {len(dtc_rows)} DTC character area rows.")
pd.DataFrame(dtc_rows)[["district_code","dtc_character_area","multifamily_permitted",
                          "mf_conversion_flag","max_height_ft"]]

Built 15 DTC character area rows.


,district_code,dtc_character_area,multifamily_permitted,mf_conversion_flag,max_height_ft
0,DTC-BM,Biomed,True,True,None
1,DTC-BC,Business Core,True,True,None
2,DTC-CP,Central Park,True,True,None
3,DTC-CC,Commercial Corridors,True,True,None
4,DTC-DG,Downtown Gateway,True,True,None
5,DTC-EE,East Evergreen,True,True,None
6,DTC-ECE,Evans Churchill East,True,True,None
7,DTC-ECW,Evans Churchill West,True,True,None
8,DTC-MC,McDowell Corridor,True,True,None
9,DTC-RE,Roosevelt East,True,True,None


## Section 4 — Land Use Matrix → `permits_*` columns

In [85]:
def parse_land_use_matrix() -> pd.DataFrame:
    """
    Parse Section 608.D land use matrix.
    Returns a DataFrame indexed by use_label with district columns.
    """
    sec = soup.find("section", id="ZO_608.D")
    if not sec:
        raise ValueError("ZO_608.D not found")
    
    table = sec.find("table")
    if not table:
        raise ValueError("No table in ZO_608.D")
    
    rows = table.find_all("tr")
    
    # Row 0: header (LAND USE CATEGORIES | SECTION AND ZONING DISTRICT)
    # Row 1: section numbers (609, 610, ...) — skip
    # Row 2: district codes (RE-35, R1-18, ...) — use as column names
    # Row 3+: data rows

    district_row = rows[2]
    districts = [c.get_text(strip=True) for c in district_row.find_all(["td","th"])]
    # First two cells are category columns; rest are district codes
    district_cols = districts  # keep full list, we'll handle offset per row

    records = []
    for row in rows[3:]:
        cells = row.find_all(["td", "th"])
        texts = [c.get_text(strip=True) for c in cells]
        if len(texts) < 3:
            continue
        
        # Handle (ac) prefix rows — accessory use rows have 3 label cells
        if texts[0] == "(ac)":
            use_label = texts[0] + " " + texts[1]
            values    = texts[2:]
        else:
            use_label = texts[0]
            values    = texts[1:]
        
        # Align values to district columns
        rec = {"use_label": use_label}
        dist_codes = ["RE-35","R1-18","R1-10","R1-8","R1-6",
                      "R-2","R-3","R-3A","R-4","R-5","R-4A","PAD"]
        for i, code in enumerate(dist_codes):
            rec[code] = values[i].strip() if i < len(values) else ""
        records.append(rec)
    
    return pd.DataFrame(records).set_index("use_label")


lum = parse_land_use_matrix()
print(f"Land use matrix: {len(lum)} rows × {len(lum.columns)} district columns")
print("\nDistrict columns:", lum.columns.tolist())
print("\nFirst 10 use labels:")
for label in lum.index[:10]:
    print(" ", label)

Land use matrix: 39 rows × 12 district columns

District columns: ['RE-35', 'R1-18', 'R1-10', 'R1-8', 'R1-6', 'R-2', 'R-3', 'R-3A', 'R-4', 'R-5', 'R-4A', 'PAD']

First 10 use labels:
  Single-Family Detached
  (ac) Accessory Dwelling Unit(ADU)
  (ac) Guestroom(s)
  Single-Family Attached
  Single-Family Infill (SFI) Subdivision
  Multi-Family
  (ac) Residential Convenience Market
  Off-Site Manufactured Home Developments
  Boarding House
  1-4Adult Day Care Home


In [86]:
def is_permitted(code: str) -> bool:
    """
    Return True if the land use matrix code represents a permitted use.
    Permitted = 'p' or 'pc' (conditionally permitted without a public hearing).
    Not permitted = 'np'.
    Use permit required = 'up' — treated as False for MMH feasibility
    because use permits require discretionary approval.
    """
    c = code.lower().strip()
    if c == "np" or c == "":
        return False
    if c.startswith("up"):
        return False   # Discretionary; not by-right
    if c.startswith("p"):  # 'p' or 'pc...' 
        return True
    return False


# Multi-family label in the matrix — confirm the exact string
mf_label = [l for l in lum.index if "Multi-Family" in l or "multi-family" in l.lower()]
print("Multi-family labels found:", mf_label)

# Show a relevant slice of the matrix
relevant_uses = [
    l for l in lum.index
    if any(kw in l.lower() for kw in ["multi", "single-family infill", "multi-family"])
]
print("\nRelevant use rows:")
lum.loc[relevant_uses]

Multi-family labels found: ['Multi-Family']

Relevant use rows:


,RE-35,R1-18,R1-10,R1-8,R1-6,R-2,R-3,R-3A,R-4,R-5,R-4A,PAD
use_label,,,,,,,,,,,,
Single-Family Infill (SFI) Subdivision,np,np,np,np,np,pc5up5,pc5up5,pc5up5,pc5up5,pc5up5,pc6up6,np
Multi-Family,np,np,np,np,np,pc7,pc7,pc7,pc7,pc7,pc7,pc7


In [87]:
# ── Assign permits_* for all rows via land use matrix + hardcoded logic ──────────

MULTIFAMILY_LABEL = "Multi-Family"
DISTRICT_ORDER    = ["RE-35","R1-18","R1-10","R1-8","R1-6",
                     "R-2","R-3","R-3A","R-4","R-5","R-4A"]
COTTAGE_CLUSTER_START_IDX = 6  # R-3 is first cottage-eligible

# Districts already hardcoded during row construction — skip matrix lookup
_SKIP_CODES = (
    {"C-1","C-2","C-3","UR","C-O","R-O","R-I","H-R","PUD","WU","PC","PSC"}
    | {r["district_code"] for r in dtc_rows}
)

def get_permits_for_district(district_code: str) -> dict:
    """
    Return permits_* dict for a district code by reading the §608.D matrix.
    Returns None for all permits for unrecognized/overlay districts.
    """
    if district_code not in lum.columns:
        return {
            "permits_duplex": None, "permits_triplex": None,
            "permits_fourplex": None, "permits_courtyard_apt": None,
            "permits_cottage_cluster": None, "multifamily_permitted": None,
        }

    mf_code = ""
    for label in lum.index:
        if MULTIFAMILY_LABEL in label and "(ac)" not in label:
            mf_code = lum.loc[label, district_code]
            break

    mf_permitted  = is_permitted(mf_code)
    district_idx  = DISTRICT_ORDER.index(district_code) if district_code in DISTRICT_ORDER else -1
    cottage_ok    = mf_permitted and district_idx >= COTTAGE_CLUSTER_START_IDX

    return {
        "permits_duplex":          mf_permitted,
        "permits_triplex":         mf_permitted,
        "permits_fourplex":        mf_permitted,
        "permits_courtyard_apt":   mf_permitted,
        "permits_cottage_cluster": cottage_ok,
        "multifamily_permitted":   mf_permitted,
    }


# Apply to prose and table rows (not commercial/overlay/DTC which are hardcoded)
for rows_list in [prose_rows, table_rows]:
    for row in rows_list:
        if row["district_code"] not in _SKIP_CODES:
            row.update(get_permits_for_district(row["district_code"]))

# Spot-check
assert get_permits_for_district("R1-6")["multifamily_permitted"]   == False,  "R1-6 MF should be False"
assert get_permits_for_district("R-2")["multifamily_permitted"]    == True,   "R-2 MF should be True"
assert get_permits_for_district("R-2")["permits_cottage_cluster"]  == False,  "R-2 cottage should be False"
assert get_permits_for_district("R-3")["permits_cottage_cluster"]  == True,   "R-3 cottage should be True"
assert get_permits_for_district("RE-35")["multifamily_permitted"]  == False,  "RE-35 MF should be False"
print("Land use matrix permits assigned and validated.")

Land use matrix permits assigned and validated.


### ⚠️ Manual review checkpoint — Section 4

Verify the `permits_*` values are as expected:
- RE-35 through R1-6: all `permits_*` = `False`
- R-2: `permits_duplex/triplex/fourplex/courtyard_apt` = `True`; `permits_cottage_cluster` = `False`
- R-3 and above: all `permits_*` = `True`
- C-1/C-2/C-3: hardcoded `True` (not from matrix — residential governed by R-3 standards)
- R-O/C-O: hardcoded `False` (residential accessory only or limited)

## Section 5 — Parking table → `parking_min_per_du`

In [88]:
def parse_parking_table() -> dict:
    """
    Parse Section 702.C parking requirements table.
    Returns a dict mapping simplified use category → (spaces_per_du, notes).
    """
    sec = soup.find(id="ZO_702")
    if not sec:
        raise ValueError("ZO_702 not found")
    
    # Table 0 in 702 is the main parking requirements table
    tables = sec.find_all("table")
    parking_table = tables[0]
    
    rows = parking_table.find_all("tr")
    result = {}
    for row in rows[1:]:  # skip header
        cells = row.find_all(["td","th"])
        if len(cells) < 2:
            continue
        use  = cells[0].get_text(strip=True)
        reqs = cells[1].get_text(strip=True)
        result[use] = reqs
    
    return result


parking_raw = parse_parking_table()

# Print only the dwelling unit rows
dw_rows = {k: v for k, v in parking_raw.items() if "Dwelling" in k}
for k, v in dw_rows.items():
    print(f"  {k}")
    print(f"    → {v}")

  Dwelling Unit, Multi-family
    → 1.5 spaces perdwelling unitOf the required parking spaces, a minimum of 0.25 spaces perdwelling unitmust be provided as unreserved spaces when thelothas five or moredwelling units.
  Dwelling Unit, Single-FamilyAttached
    → 2 spaces perprimary dwelling unit. The required spaces for eachdwelling unitmust be provided on the samelot. An additional 0.25 unreserved spaces perdwelling unitmust be provided elsewhere within the development for visitor parking.
  Dwelling Unit, Single-FamilyDetached
    → 2 spaces per primarydwelling unit


In [89]:
# Extract baseline parking minimums from §702.C
PARKING_SINGLE_FAMILY_DETACHED_KEY = "Dwelling Unit, Single-Family Detached"
PARKING_MULTIFAMILY_KEY            = "Dwelling Unit, Multi-family"

parking_sf_detached = extract_ft(parking_raw.get(PARKING_SINGLE_FAMILY_DETACHED_KEY, ""))
parking_multifamily = extract_ft(parking_raw.get(PARKING_MULTIFAMILY_KEY, ""))

print(f"Single-family detached: {parking_sf_detached} spaces/DU   (§702.C)")
print(f"Multi-family:           {parking_multifamily} spaces/DU   (§702.C)")
print()
print("MH Overlay parking (§632.E.5):")
print("  ≤1 PDU:  2 spaces on site")
print("  ≥2 PDUs: 1 space per PDU on site")
print()
print("UR parking (§642.I.1) — hardcoded in ur_row:")
print("  1BR or smaller: 1 space/DU")
print("  2+BR:           1.5 spaces/DU")

Single-family detached: None spaces/DU   (§702.C)
Multi-family:           1.5 spaces/DU   (§702.C)

MH Overlay parking (§632.E.5):
  ≤1 PDU:  2 spaces on site
  ≥2 PDUs: 1 space per PDU on site

UR parking (§642.I.1) — hardcoded in ur_row:
  1BR or smaller: 1 space/DU
  2+BR:           1.5 spaces/DU


## Section 6 — Special and placeholder rows (PUD, WU Code, PC, PSC)

In [90]:
BLANK_ROW_TEMPLATE = {
    "max_density_du_per_acre":           None,
    "max_density_notes":                  "",
    "max_height_ft":                      None,
    "max_height_stories":                 None,
    "min_lot_width_ft":                   None,
    "min_lot_depth_ft":                   None,
    "max_lot_coverage_pct":               None,
    "adu_coverage_bonus_pct":             None,
    "min_front_setback_ft":               None,
    "min_rear_setback_ft":                None,
    "min_street_side_setback_ft":         None,
    "min_interior_side_setback_a_ft":     None,
    "min_interior_side_setback_b_ft":     None,
    "max_density_du_per_acre_prd":        None,
    "max_height_ft_prd":                  None,
    "max_height_stories_prd":             None,
    "max_lot_coverage_pct_prd":           None,
    "parking_min_per_du":                 None,
    "parking_notes":                      "",
    "permits_duplex":                     None,
    "permits_triplex":                    None,
    "permits_fourplex":                   None,
    "permits_courtyard_apt":              None,
    "permits_cottage_cluster":            None,
    "multifamily_permitted":              None,
    "pud_flag":                           False,
    "wu_code_flag":                       False,
    "dtc_flag":                           False,
    "dtc_character_area":                 None,
    "mf_conversion_flag":                 False,
    "ri_overlay_eligible":                False,
    "hr_overlay_eligible":                False,
    "mh_overlay_eligible":            False,
    "hp_overlay_eligible":            False,
    "notes":                              "",
}

pud_row = {
    **BLANK_ROW_TEMPLATE,
    "district_code":    "PUD",
    "district_name":    "Planned Unit Development",
    "ordinance_section": "Section 671",
    "district_type":    "planned_development",
    "pud_flag":         True,
    "notes":            "Standards negotiated per development plan (§671). All numeric fields NULL.",
}

wu_row = {
    **BLANK_ROW_TEMPLATE,
    "district_code":    "WU",
    "district_name":    "Walkable Urban Code (placeholder row)",
    "ordinance_section": "Chapter 13",
    "district_type":    "form_based_code",
    "wu_code_flag":     True,
    "notes":            "WU Code standards by transect — see wu_transect_standards.csv (Section 8). "
                        "Applies only within approved TOD district boundaries per §1301.C map.",
}

pc_row = {
    **BLANK_ROW_TEMPLATE,
    "district_code":    "PC",
    "district_name":    "Planned Community District",
    "ordinance_section": "Section 636",
    "district_type":    "planned_development",
    "pud_flag":         True,
    "notes":            "Large-scale planned community (§636). Standards per master plan. All numeric fields NULL.",
}

psc_row = {
    **BLANK_ROW_TEMPLATE,
    "district_code":    "PSC",
    "district_name":    "Planned Shopping Center District",
    "ordinance_section": "Section 637",
    "district_type":    "commercial",
    "multifamily_permitted": False,
    "notes":            "Planned shopping center (§637). Residential uses not typical. "
                        "§711 may apply if an existing commercial building in a PSC qualifies as obsolete.",
}

special_rows = [pud_row, wu_row, pc_row, psc_row]
print(f"Special rows built: {[r['district_code'] for r in special_rows]}")

Special rows built: ['PUD', 'WU', 'PC', 'PSC']


## Section 3e — Remaining overlay districts and special zones

Adds rows for:
- **MH** (§632) — Middle Housing Overlay: up to 4 PDUs per lot on SF-zoned land
- **HP** (§802–815) — Historic Preservation suffix: ARB review + demolition restriction
- **AIO** (§644) — Airport Noise Impact Overlay (Sky Harbor vicinity)
- **DVAO** (§658) — Deer Valley Airport Overlay
- **TOD-1** (§662) — Transit-Oriented Development Overlay District One (LRT corridor)
- **TOD-2** (§663) — Transit-Oriented Development Overlay District Two (LRT corridor)
- **FHEM** (§657) — Flood Hazard and Erosion Management District

Also patches two new flag columns onto all existing row groups:
- `mh_overlay_eligible` — base districts eligible to receive the MH Overlay designation
- `hp_overlay_eligible` — districts where HP designation is possible (City Council action)

**Overlays NOT found in the ordinance HTML:**
- §625 does not exist (gap between §624 and §626). The "TOC" referred to in planning
  documents is the Transit-Oriented Corridor Plan — a policy document, not a zoning overlay.
  The zoning equivalents are TOD-1 (§662) and TOD-2 (§663).
- No discrete "Hillside" overlay district exists in the ordinance. The Desert Character
  Overlay (§653) addresses terrain-sensitive development in north Phoenix, but is not
  a hillside code. The Sonoran Preserve Overlay (§654) addresses preserve adjacency.
- No "Freeway Frontage District" exists as a standalone district; freeway-adjacent parcels
  are governed by corridor-specific overlays (§651, §660, §665, §666) or base zoning.

In [91]:
# ── MH Overlay (§632) — Middle Housing Overlay ───────────────────────────────
#
# Key provisions (from §632.B–E):
#   - Permits up to 4 PDUs per lot (overrides base district maximum if base allows fewer)
#   - Setbacks/height: same as a single-family DU on the same property
#   - Lot coverage: same as base district (exception: RE-43/RE-35/RE-24/R1-14 → 50% max
#     if ALL structures are 1-story and ≤20 ft height)
#   - Parking: ≤1 PDU = 2 spaces on site; ≥2 PDUs = 1 space per PDU
#   - Maneuvering within a public alley is allowed
#
# Eligibility (§632.C):
#   Option 1: City Council rezoning with MH Overlay designation (any eligible base zone)
#   Option 2: Opt-in for qualifying new subdivisions (≥10 gross acres, platted ≥Jan 1 2026,
#              max 20% of lots, base zone from eligible list below)
#   Eligible base zones: RE-43, RE-24, R1-14, RE-35, R1-18, R1-10, R1-8, R1-6,
#                        R-2, R-3, R-3A, R-4, R-5, R-4A, C-1, C-2, C-3
#   (For C-1/C-2/C-3: only when platted as SFI subdivision per §608.I)
#
# INELIGIBLE near:
#   - FAA commercially licensed airports (Sky Harbor, Deer Valley, Scottsdale Airport)
#   - Military installation AICUZ noise ≥65 dB DNL
#   - Proposed freeway ≤600 ft from lot
#   - Superfund/WQARF sites
#   - State trust land (not subdivided)

# Eligible base zones for mh_overlay_eligible flag
_MH_ELIGIBLE_BASE_ZONES = {
    "RE-43","RE-24","R1-14","RE-35","R1-18","R1-10","R1-8","R1-6",
    "R-2","R-3","R-3A","R-4","R-5","R-4A",
    "C-1","C-2","C-3",  # only when SFI platted
}

mh_row = {
    "district_code":                  "MH",
    "district_name":                  "Middle Housing Overlay",
    "ordinance_section":              "Section 632",
    "district_type":                  "overlay",
    "max_density_du_per_acre":        None,     # Up to 4 PDUs/lot; lot count/acre from base district
    "max_density_notes":              (
        "Max 4 PDUs per lot (or base district max, whichever is greater). "
        "Lot count per acre unchanged from base district. "
        "PRD density bonus (§608.H.1) applies for SF development on MH lots. "
        "Base zone max subdivided lots/acre is not modified by MH Overlay."
    ),
    "max_height_ft":                  None,     # Same as SF DU per base district
    "max_height_stories":             None,
    "min_lot_width_ft":               None,     # Same as base district
    "min_lot_depth_ft":               None,
    "max_lot_coverage_pct":           None,     # Same as base (see RE-43/RE-35/RE-24/R1-14 exception)
    "adu_coverage_bonus_pct":         None,
    "min_front_setback_ft":           None,     # Same as SF DU setbacks per base district
    "min_rear_setback_ft":            None,
    "min_street_side_setback_ft":     None,
    "min_interior_side_setback_a_ft": None,
    "min_interior_side_setback_b_ft": None,
    "max_density_du_per_acre_prd":    None,
    "max_height_ft_prd":              None,
    "max_height_stories_prd":         None,
    "max_lot_coverage_pct_prd":       None,
    "parking_min_per_du":             1.0,      # ≤1 PDU: 2 spaces total; ≥2 PDUs: 1/PDU
    "parking_notes":                  (
        "≤1 PDU: 2 spaces minimum on site (§632.E.5.a). "
        "≥2 PDUs: 1 space per PDU on site (§632.E.5.b). "
        "Maneuvering within public alley permitted. "
        "Tandem parking and uncovered spaces allowed."
    ),
    "permits_duplex":                 True,
    "permits_triplex":                True,
    "permits_fourplex":               True,     # Up to 4 PDUs by right
    "permits_courtyard_apt":          True,
    "permits_cottage_cluster":        True,
    "multifamily_permitted":          True,
    "pud_flag":                       False,
    "wu_code_flag":                   False,
    "dtc_flag":                       False,
    "dtc_character_area":             None,
    "mf_conversion_flag":             False,
    "ri_overlay_eligible":            False,
    "hr_overlay_eligible":            False,
    "mh_overlay_eligible":            True,
    "hp_overlay_eligible":            False,
    "notes": (
        "Middle Housing Overlay implements ARS §9-462.13 (§632). "
        "Permits up to 4 PDUs per lot; multi-family DUs explicitly permitted (§632.E.1). "
        "Setbacks/height: same as SF DU on same property (§632.E.3). "
        "RE-43/RE-35/RE-24/R1-14 lot coverage: up to 50% when all structures ≤1 story and ≤20 ft. "
        "Eligibility: rezoning OR opt-in for qualifying new subdivisions (≥10 acres, platted ≥Jan 1 2026, "
        "max 20% of lots; base zone from §632.C eligible list). "
        "INELIGIBLE near: Sky Harbor/Deer Valley/Scottsdale Airport; military AICUZ ≥65 dB; "
        "proposed freeway ≤600 ft; Superfund/WQARF sites; unsubdivided state trust land."
    ),
}

print("MH row built:")
for k, v in mh_row.items():
    if v not in (None, False, ""):
        print(f"  {k}: {v!r}")

MH row built:
  district_code: 'MH'
  district_name: 'Middle Housing Overlay'
  ordinance_section: 'Section 632'
  district_type: 'overlay'
  max_density_notes: 'Max 4 PDUs per lot (or base district max, whichever is greater). Lot count per acre unchanged from base district. PRD density bonus (§608.H.1) applies for SF development on MH lots. Base zone max subdivided lots/acre is not modified by MH Overlay.'
  parking_min_per_du: 1.0
  parking_notes: '≤1 PDU: 2 spaces minimum on site (§632.E.5.a). ≥2 PDUs: 1 space per PDU on site (§632.E.5.b). Maneuvering within public alley permitted. Tandem parking and uncovered spaces allowed.'
  permits_duplex: True
  permits_triplex: True
  permits_fourplex: True
  permits_courtyard_apt: True
  permits_cottage_cluster: True
  multifamily_permitted: True
  mh_overlay_eligible: True
  notes: 'Middle Housing Overlay implements ARS §9-462.13 (§632). Permits up to 4 PDUs per lot; multi-family DUs explicitly permitted (§632.E.1). Setbacks/height: same as

In [92]:
# ── HP Historic Preservation (Chapter 8, §802–815) ───────────────────────────
#
# The HP suffix is applied by City Council to specific properties/districts.
# It modifies procedural requirements but does NOT modify base district
# density/height/setback standards — those remain from the underlying zone.
#
# Key constraints (from §811–814):
#   - Any demolition requires Demolition Approval from HP Officer, HP Commission,
#     or City Council (§813)
#   - Any construction, alteration, remodeling, or landscaping requires a
#     Certificate of No Effect OR Certificate of Appropriateness from HP Officer
#     or HP Commission (§812)
#   - Economic hardship may be claimed to justify relaxation of restrictions (§814)
#
# From §806: Interim protection — demolition moratorium applies even during
# application for HP designation (before City Council action).
#
# From §815: Phoenix Historic Property Register tracks designated sites.
#
# Scanner implications:
#   - HP parcels have a demolition barrier (major constraint on redevelopment)
#   - ARB/HP Commission review required for ANY development (time/cost increase)
#   - hp_overlay_eligible = True on all residential/commercial base district rows

hp_row = {
    "district_code":                  "HP",
    "district_name":                  "Historic Preservation Overlay",
    "ordinance_section":              "Chapter 8 / Sections 802–815",
    "district_type":                  "overlay",
    "max_density_du_per_acre":        None,     # From underlying base district
    "max_density_notes":              (
        "HP does not modify density. Base district density applies. "
        "Redevelopment is effectively constrained by demolition approval requirement."
    ),
    "max_height_ft":                  None,     # From underlying base district
    "max_height_stories":             None,
    "min_lot_width_ft":               None,
    "min_lot_depth_ft":               None,
    "max_lot_coverage_pct":           None,
    "adu_coverage_bonus_pct":         None,
    "min_front_setback_ft":           None,
    "min_rear_setback_ft":            None,
    "min_street_side_setback_ft":     None,
    "min_interior_side_setback_a_ft": None,
    "min_interior_side_setback_b_ft": None,
    "max_density_du_per_acre_prd":    None,
    "max_height_ft_prd":              None,
    "max_height_stories_prd":         None,
    "max_lot_coverage_pct_prd":       None,
    "parking_min_per_du":             None,     # Per underlying base district
    "parking_notes":                  "Per underlying base district. ARB may require design accommodations.",
    "permits_duplex":                 None,     # Depends on base district
    "permits_triplex":                None,
    "permits_fourplex":               None,
    "permits_courtyard_apt":          None,
    "permits_cottage_cluster":        None,
    "multifamily_permitted":          None,
    "pud_flag":                       False,
    "wu_code_flag":                   False,
    "dtc_flag":                       False,    # DTC has separate HP provisions (§1201.B.2)
    "dtc_character_area":             None,
    "mf_conversion_flag":             False,    # §711 conversion likely incompatible with HP designation
    "ri_overlay_eligible":            False,
    "hr_overlay_eligible":            False,    # H-R height increases typically incompatible with HP
    "mh_overlay_eligible":            False,    # HP parcels excluded from MH Overlay by implication
    "hp_overlay_eligible":            True,
    "notes": (
        "HP suffix applied by City Council designation (§807). "
        "Does not modify base district density/height/setbacks — adds procedural constraints only. "
        "DEMOLITION: requires Demolition Approval from HP Officer, HP Commission, or City Council (§813). "
        "ALL development (construction, alteration, remodeling, landscaping): requires "
        "Certificate of No Effect or Certificate of Appropriateness from HP Officer or Commission (§812). "
        "Interim protection: demolition moratorium applies during pending HP designation application (§806). "
        "Economic hardship provisions available for income-producing properties (§814). "
        "DTC parcels with HP zoning: HP Chapter 8 prevails over DTC standards (§1201.B.2). "
        "WU Code parcels: HP adjacency setback rules apply (§1303.A.3). "
        "Scanner impact: hp_flag = TRUE is a strong redevelopment constraint; "
        "MH Overlay and H-R Overlay eligibility effectively blocked."
    ),
}

print("HP row built.")

HP row built.


In [93]:
# ── AIO — Airport Noise Impact Overlay (§644, Sky Harbor) ────────────────────
#
# Geography: Southern Pacific Railroad (N) / University Dr (S) / 7th St (W) /
#             16th St and 24th St (E) — south-central Phoenix near Sky Harbor
# Prohibits: off-site constructed DUs (manufactured/modular homes)
# Requires: noise notice recorded with County Recorder before building permit issuance
# Design review: all new homes subject to §507 Tab A design review
# Additional standards for 1–2 unit residential:
#   - All driveways/parking must be hard surface
#   - Each DU must have ≥1 covered parking space (garage or carport)
#   - Front yard must be landscaped (for 2-unit); rear yard landscaping required
#
# Note: MH Overlay EXPLICITLY EXCLUDES properties near Sky Harbor (§632.C.2.f)

aio_row = {
    "district_code":                  "AIO",
    "district_name":                  "Airport Noise Impact Overlay — Sky Harbor",
    "ordinance_section":              "Section 644",
    "district_type":                  "overlay",
    "max_density_du_per_acre":        None,     # Per underlying base district
    "max_density_notes":              (
        "Density per underlying base district. "
        "Off-site constructed DUs (manufactured/modular homes) prohibited (§644.C). "
        "MH Overlay INELIGIBLE within AIO boundary (§632.C.2.f)."
    ),
    "max_height_ft":                  None,
    "max_height_stories":             None,
    "min_lot_width_ft":               None,
    "min_lot_depth_ft":               None,
    "max_lot_coverage_pct":           None,
    "adu_coverage_bonus_pct":         None,
    "min_front_setback_ft":           None,
    "min_rear_setback_ft":            None,
    "min_street_side_setback_ft":     None,
    "min_interior_side_setback_a_ft": None,
    "min_interior_side_setback_b_ft": None,
    "max_density_du_per_acre_prd":    None,
    "max_height_ft_prd":              None,
    "max_height_stories_prd":         None,
    "max_lot_coverage_pct_prd":       None,
    "parking_min_per_du":             None,     # Per base district + must include ≥1 covered space/DU
    "parking_notes":                  (
        "Per base district. Additional: each DU must have ≥1 covered parking space "
        "(garage or carport, design matching housing materials) (§644.E.3.b). "
        "All driveways/parking spaces must be hard surface (§644.E.3.a)."
    ),
    "permits_duplex":                 None,     # Per base district (not modified by AIO)
    "permits_triplex":                None,
    "permits_fourplex":               None,
    "permits_courtyard_apt":          None,
    "permits_cottage_cluster":        None,
    "multifamily_permitted":          None,
    "pud_flag":                       False,
    "wu_code_flag":                   False,
    "dtc_flag":                       False,
    "dtc_character_area":             None,
    "mf_conversion_flag":             False,
    "ri_overlay_eligible":            False,
    "hr_overlay_eligible":            False,
    "mh_overlay_eligible":            False,    # Explicitly excluded by §632.C.2.f
    "hp_overlay_eligible":            True,     # HP can apply within AIO geography
    "notes": (
        "Airport Noise Impact Overlay — Sky Harbor International Airport vicinity (§644). "
        "Boundary: Southern Pacific Railroad / University Dr / 7th St / 16th+24th St. "
        "Applies to new residential uses established after ordinance effective date. "
        "PROHIBITED: off-site constructed DUs (manufactured homes, modular construction). "
        "REQUIRED before building permit: noise notice recorded with Maricopa County Recorder (§644.E.1). "
        "REQUIRED: §507 Tab A design review for ALL new homes (§644.E.2). "
        "ADDITIONAL STANDARDS for 1–2 unit residential: "
        "hard surface driveway/parking; ≥1 covered parking space per DU (garage/carport); "
        "front yard landscaped and separated from parking (2-unit); rear yard landscaping. "
        "MH Overlay INELIGIBLE within this boundary (§632.C.2.f). "
        "Scanner: identify AIO parcels via supplementary zoning map — flag as mh_ineligible."
    ),
}

# ── DVAO — Deer Valley Airport Overlay (§658) ─────────────────────────────────
#
# Geography: Happy Valley Rd (N) / 29th-31st Ave (W) / Rose Garden Ln (S) /
#             Cave Creek Rd / CAP Canal / 16th-20th St alignment (E)
# 3 regulation areas with escalating restrictions:
#   Area 1: No residential in A-1 zone (1 DU as watchman/caretaker accessory only)
#   Areas 2&3: All Area 1 restrictions + NO residential in C-1/C-2/C-3;
#              no day care centers, assembly halls, churches, community residences,
#              dependent care facilities, group homes, hospitals, nursery schools,
#              nursing homes, schools (private)
#   Area 3: Height restrictions based on distance from airport runway centerline
# MH Overlay INELIGIBLE near Deer Valley (§632.C.2.f)

dvao_row = {
    "district_code":                  "DVAO",
    "district_name":                  "Deer Valley Airport Overlay",
    "ordinance_section":              "Section 658",
    "district_type":                  "overlay",
    "max_density_du_per_acre":        None,
    "max_density_notes":              (
        "Density per underlying base district in Areas 2/3 (no residential in A-1 zone). "
        "Areas 2+3: residential PROHIBITED in C-1/C-2/C-3 (except 1 DU for on-site caretaker). "
        "Area 3: height restrictions apply based on distance from runway centerline (Figure 1). "
        "MH Overlay INELIGIBLE within DVAO boundary (§632.C.2.f)."
    ),
    "max_height_ft":                  None,     # Area 3: varies by distance from runway
    "max_height_stories":             None,
    "min_lot_width_ft":               None,
    "min_lot_depth_ft":               None,
    "max_lot_coverage_pct":           None,
    "adu_coverage_bonus_pct":         None,
    "min_front_setback_ft":           None,
    "min_rear_setback_ft":            None,
    "min_street_side_setback_ft":     None,
    "min_interior_side_setback_a_ft": None,
    "min_interior_side_setback_b_ft": None,
    "max_density_du_per_acre_prd":    None,
    "max_height_ft_prd":              None,
    "max_height_stories_prd":         None,
    "max_lot_coverage_pct_prd":       None,
    "parking_min_per_du":             None,
    "parking_notes":                  "Per underlying base district",
    "permits_duplex":                 None,     # Depends on area and underlying zone
    "permits_triplex":                None,
    "permits_fourplex":               None,
    "permits_courtyard_apt":          None,
    "permits_cottage_cluster":        None,
    "multifamily_permitted":          None,
    "pud_flag":                       False,
    "wu_code_flag":                   False,
    "dtc_flag":                       False,
    "dtc_character_area":             None,
    "mf_conversion_flag":             False,
    "ri_overlay_eligible":            False,
    "hr_overlay_eligible":            False,
    "mh_overlay_eligible":            False,    # Explicitly excluded by §632.C.2.f
    "hp_overlay_eligible":            True,
    "notes": (
        "Deer Valley Airport Overlay — DVA general aviation facility (§658). "
        "Boundary: Happy Valley Rd / 29th-31st Ave / Rose Garden Ln / Cave Creek Rd+CAP Canal / 16th-20th St. "
        "Applies to all new uses/development after Feb 14, 2007. "
        "THREE REGULATION AREAS (most to least restrictive): "
        "Area 1: No residential in A-1 zone (1 caretaker DU as accessory to industrial/ag only). "
        "Areas 2+3: Area 1 restrictions + residential PROHIBITED in C-1/C-2/C-3; "
        "no day care, assembly halls, churches, community residences, group homes, "
        "hospitals, nursing homes, nursery schools, private schools. "
        "Area 3: Height restrictions per distance from runway centerline (Figure 1 — in image). "
        "Required: avigation notice recorded before building permit for >2,000 sqft construction (§658.D). "
        "MH Overlay INELIGIBLE within this boundary (§632.C.2.f). "
        "Scanner: identify DVAO parcels via supplementary zoning map 1116."
    ),
}

airport_rows = [aio_row, dvao_row]
print(f"Built {len(airport_rows)} airport overlay rows: {[r['district_code'] for r in airport_rows]}")

Built 2 airport overlay rows: ['AIO', 'DVAO']


In [94]:
# ── TOD-1 (§662) and TOD-2 (§663) — Transit-Oriented Development Overlays ─────
#
# Both overlays apply to lands on Supplementary Zoning Map 1086 (Nov 19, 2003)
# along the Central Phoenix/East Valley LRT corridor.
#
# TOD-1 (§662):
#   - Strongest restrictions: prohibits gas stations, car washes, drive-ins, bulk retail,
#     exterior storage, manufactured home sales, RV parks, etc.
#   - Parking MAXIMUM replaces minimum (caps auto-parking to encourage transit use)
#   - TOD-1 standards OVERRIDE conflicting underlying district standards (§662.C)
#   - Key for scanner: reduces or eliminates parking minimums; may increase density flexibility
#
# TOD-2 (§663):
#   - Less restrictive: prohibits car washes, cemeteries, drive-ins, exterior displays,
#     golf courses, RV parks only
#   - Uses requiring permits in TOD-1 are by-right in TOD-2 (if along LRT street)
#   - TOD-2 standards override conflicting underlying standards (§663.C)
#
# Scanner note: §625 does NOT exist in the ordinance HTML. The "TOC" Transit-Oriented
# Corridor Plan is a policy document. TOD-1/TOD-2 (§662-663) are the applicable zoning overlays.

tod1_row = {
    "district_code":                  "TOD-1",
    "district_name":                  "Transit-Oriented Development Overlay District One",
    "ordinance_section":              "Section 662",
    "district_type":                  "overlay",
    "max_density_du_per_acre":        None,     # Per underlying base district (not modified by TOD-1)
    "max_density_notes":              (
        "Density per underlying base district. "
        "TOD-1 does not set density limits but requires compact form, pedestrian orientation, "
        "and minimum 1 employee/user per 500 sqft for use permits (§662.F.2.a). "
        "TOD-1 standards override conflicting underlying district standards (§662.C)."
    ),
    "max_height_ft":                  None,
    "max_height_stories":             None,
    "min_lot_width_ft":               None,
    "min_lot_depth_ft":               None,
    "max_lot_coverage_pct":           None,
    "adu_coverage_bonus_pct":         None,
    "min_front_setback_ft":           None,
    "min_rear_setback_ft":            None,
    "min_street_side_setback_ft":     None,
    "min_interior_side_setback_a_ft": None,
    "min_interior_side_setback_b_ft": None,
    "max_density_du_per_acre_prd":    None,
    "max_height_ft_prd":              None,
    "max_height_stories_prd":         None,
    "max_lot_coverage_pct_prd":       None,
    "parking_min_per_du":             None,     # Parking MAXIMUM applies, not minimum (per §662.K)
    "parking_notes":                  (
        "PARKING MAXIMUM replaces minimum within TOD-1. "
        "Excess parking requires use permit (§662.F.1.f). "
        "Parking maximum per §662.K (by floor area and land use — see Supplementary Map 1086). "
        "No standalone parking facility (commercial) permitted without use permit."
    ),
    "permits_duplex":                 None,     # Per underlying district
    "permits_triplex":                None,
    "permits_fourplex":               None,
    "permits_courtyard_apt":          None,
    "permits_cottage_cluster":        None,
    "multifamily_permitted":          None,
    "pud_flag":                       False,
    "wu_code_flag":                   False,
    "dtc_flag":                       False,
    "dtc_character_area":             None,
    "mf_conversion_flag":             True,     # §711 conversions well-suited to TOD-1 geography
    "ri_overlay_eligible":            False,
    "hr_overlay_eligible":            False,
    "mh_overlay_eligible":            True,     # MH Overlay CAN apply within TOD-1 (no exclusion)
    "hp_overlay_eligible":            True,
    "notes": (
        "Transit-Oriented Development Overlay District One — Central Phoenix/East Valley LRT corridor (§662). "
        "Applies to Supplementary Zoning Map 1086 (adopted Nov 19, 2003). "
        "PROHIBITED uses: gas stations, car washes, drive-in businesses, bulk retail, "
        "exterior storage, manufactured home sales, RV parks, towing, truck stops, boat dealers (§662.D). "
        "PARKING: maximum (not minimum) applies — excess parking requires use permit (§662.F.1.f). "
        "TOD-1 OVERRIDES conflicting underlying district standards (§662.C). "
        "Use permits required for: grocery stores >50,000 sqft, light industrial, liquor retail, "
        "outdoor dining with alcohol, outdoor recreation, commercial parking (§662.F). "
        "Large retail/mixed-use centers ≥40 acres within village cores: partial TOD-1 exemption "
        "for redevelopment not exceeding 30% increase in existing building footprint. "
        "⚠️ §625 does NOT exist in ordinance — TOD-1/TOD-2 are the applicable transit overlay codes."
    ),
}

tod2_row = {
    "district_code":                  "TOD-2",
    "district_name":                  "Transit-Oriented Development Overlay District Two",
    "ordinance_section":              "Section 663",
    "district_type":                  "overlay",
    "max_density_du_per_acre":        None,
    "max_density_notes":              (
        "Density per underlying base district. "
        "Less restrictive than TOD-1 — focuses on transit-adjacent properties. "
        "TOD-2 standards override conflicting underlying standards (§663.C)."
    ),
    "max_height_ft":                  None,
    "max_height_stories":             None,
    "min_lot_width_ft":               None,
    "min_lot_depth_ft":               None,
    "max_lot_coverage_pct":           None,
    "adu_coverage_bonus_pct":         None,
    "min_front_setback_ft":           None,
    "min_rear_setback_ft":            None,
    "min_street_side_setback_ft":     None,
    "min_interior_side_setback_a_ft": None,
    "min_interior_side_setback_b_ft": None,
    "max_density_du_per_acre_prd":    None,
    "max_height_ft_prd":              None,
    "max_height_stories_prd":         None,
    "max_lot_coverage_pct_prd":       None,
    "parking_min_per_du":             None,
    "parking_notes":                  (
        "Parking maximum applies within TOD-2 (per §663.K — same structure as TOD-1). "
        "Excess parking requires use permit. "
        "TOD-2 allows some uses by right along LRT streets that require permits in TOD-1."
    ),
    "permits_duplex":                 None,
    "permits_triplex":                None,
    "permits_fourplex":               None,
    "permits_courtyard_apt":          None,
    "permits_cottage_cluster":        None,
    "multifamily_permitted":          None,
    "pud_flag":                       False,
    "wu_code_flag":                   False,
    "dtc_flag":                       False,
    "dtc_character_area":             None,
    "mf_conversion_flag":             True,
    "ri_overlay_eligible":            False,
    "hr_overlay_eligible":            False,
    "mh_overlay_eligible":            True,
    "hp_overlay_eligible":            True,
    "notes": (
        "Transit-Oriented Development Overlay District Two — Central Phoenix/East Valley LRT corridor (§663). "
        "Applies to same Supplementary Zoning Map 1086 as TOD-1. "
        "PROHIBITED uses (fewer than TOD-1): car washes, cemeteries, drive-in businesses, "
        "exterior display of goods, golf courses, RV parks (§663.D). "
        "Along LRT street: auto leasing/rental and outdoor dining allowed by right "
        "(with conditions) that require permits in TOD-1 (§663.E). "
        "USE PERMITS required for: auto leasing (general), grocery >50,000 sqft, "
        "liquor retail, outdoor dining with alcohol, outdoor recreation, commercial parking (§663.F). "
        "TOD-2 OVERRIDES conflicting underlying district standards (§663.C). "
        "Less restrictive than TOD-1 overall — typically applies to properties "
        "adjacent to but not directly fronting LRT stations."
    ),
}

tod_rows = [tod1_row, tod2_row]
print(f"Built {len(tod_rows)} TOD overlay rows.")

# ── FHEM — Flood Hazard and Erosion Management District (§657) ────────────────
#
# Watercourse preservation district applied to erosion control zones.
# Residential: max 1 DU/acre — but TRANSFERABLE to adjacent residential zone.
# Non-residential: 0.1 FAR — transferable to adjacent non-residential zone.
# All structures/parking must be outside the flood zone (transferred to adjacent parcel).
# Not a typical residential development district — strong development constraint.

fhem_row = {
    "district_code":                  "FHEM",
    "district_name":                  "Flood Hazard and Erosion Management District",
    "ordinance_section":              "Section 657",
    "district_type":                  "overlay",
    "max_density_du_per_acre":        1.0,      # Max 1 DU/acre on flood zone portion (transferable)
    "max_density_notes":              (
        "Max 1 DU/acre on the FHEM-zoned portion — transferable to adjacent residential zone (§657.B.5). "
        "Non-residential: 0.1 FAR on FHEM portion, transferable to adjacent non-residential zone. "
        "All structures, parking, and accessory uses must be in the adjacent zoning district, "
        "not within the FHEM boundary."
    ),
    "max_height_ft":                  None,
    "max_height_stories":             None,
    "min_lot_width_ft":               None,
    "min_lot_depth_ft":               None,
    "max_lot_coverage_pct":           0.0,      # No structures permitted within FHEM boundary
    "adu_coverage_bonus_pct":         None,
    "min_front_setback_ft":           None,
    "min_rear_setback_ft":            None,
    "min_street_side_setback_ft":     None,
    "min_interior_side_setback_a_ft": None,
    "min_interior_side_setback_b_ft": None,
    "max_density_du_per_acre_prd":    None,
    "max_height_ft_prd":              None,
    "max_height_stories_prd":         None,
    "max_lot_coverage_pct_prd":       None,
    "parking_min_per_du":             None,
    "parking_notes":                  "Parking must be located outside FHEM boundary (transferred to adjacent zone)",
    "permits_duplex":                 False,
    "permits_triplex":                False,
    "permits_fourplex":               False,
    "permits_courtyard_apt":          False,
    "permits_cottage_cluster":        False,
    "multifamily_permitted":          False,
    "pud_flag":                       False,
    "wu_code_flag":                   False,
    "dtc_flag":                       False,
    "dtc_character_area":             None,
    "mf_conversion_flag":             False,
    "ri_overlay_eligible":            False,
    "hr_overlay_eligible":            False,
    "mh_overlay_eligible":            False,
    "hp_overlay_eligible":            False,
    "notes": (
        "Flood Hazard and Erosion Management District — watercourse preservation (§657). "
        "Applied to erosion control zones via FCDMC/City of Phoenix Master Plan Study. "
        "PERMITTED within FHEM: stormwater conveyance, water recharge, open space, "
        "hiking/riding trails, picnic areas (no game courts, no grass). "
        "RESIDENTIAL: max 1 DU/acre on FHEM portion — must be transferred to adjacent residential zone. "
        "NON-RESIDENTIAL: 0.1 FAR on FHEM portion — transferable to adjacent non-residential zone. "
        "ALL structures, parking, and accessory uses must be outside FHEM boundary. "
        "FHEM area counts toward setback/open space calculations for adjacent developed portion (§657.C.5). "
        "When density transferred: FHEM area must be dedicated as drainage/open space tract. "
        "Also subject to Phoenix City Code Article 1, Chapter 32B (Floodplains). "
        "Scanner: flag FHEM parcels as unbuildable; density transfer may add to adjacent parcel capacity."
    ),
}

print("FHEM row built.")

Built 2 TOD overlay rows.
FHEM row built.


### Patch new flag columns onto all existing row groups

`mh_overlay_eligible` and `hp_overlay_eligible` must be added to all rows built
in Sections 3–3d before assembly. The new overlay rows already have these columns.

In [95]:
# ── Patch mh_overlay_eligible and hp_overlay_eligible onto all existing rows ────
#
# mh_overlay_eligible: base districts eligible to receive MH Overlay designation
# per §632.C eligible list. C-1/C-2/C-3 only when platted as SFI subdivision.
#
# hp_overlay_eligible: HP designation can be applied to any existing district
# via City Council action. True for all residential and commercial base districts.
# False for: overlay rows, DTC rows, WU rows, special rows (PUD/PC/PSC), FHEM.

_MH_ELIGIBLE_BASE_ZONES = {
    "S-1",     # S-1 not in §632.C list — False
    "RE-43", "RE-24", "R1-14",
    "RE-35", "R1-18", "R1-10", "R1-8", "R1-6",
    "R-2", "R-3", "R-3A", "R-4", "R-5", "R-4A",
    "C-1", "C-2", "C-3",   # only when SFI platted — note in individual row
}
# S-1 is NOT in §632.C list; remove it
_MH_ELIGIBLE_BASE_ZONES.discard("S-1")

_HP_ELIGIBLE_TYPES = {"residential", "commercial", "residential_commercial"}
_HP_NOT_ELIGIBLE_CODES = {
    "R-I","H-R","MH","AIO","DVAO","TOD-1","TOD-2","FHEM","HP",
    "PUD","WU","PC","PSC",
    *[r["district_code"] for r in dtc_rows],
}

def patch_row(row):
    code = row["district_code"]
    row.setdefault("mh_overlay_eligible", code in _MH_ELIGIBLE_BASE_ZONES)
    row.setdefault("hp_overlay_eligible",
        row.get("district_type") in _HP_ELIGIBLE_TYPES and code not in _HP_NOT_ELIGIBLE_CODES
    )
    return row

for rows_list in [prose_rows, table_rows, commercial_rows, overlay_rows, dtc_rows, special_rows]:
    for row in rows_list:
        patch_row(row)

# Verify key expectations
_re35 = next(r for r in table_rows if r["district_code"] == "RE-35")
_s1   = next(r for r in prose_rows if r["district_code"] == "S-1")
_c1   = next(r for r in commercial_rows if r["district_code"] == "C-1")
_ri   = next(r for r in overlay_rows if r["district_code"] == "R-I")
assert _re35["mh_overlay_eligible"] == True,  "RE-35 should be MH eligible"
assert _s1["mh_overlay_eligible"]   == False, "S-1 should NOT be MH eligible"
assert _c1["mh_overlay_eligible"]   == True,  "C-1 should be MH eligible"
assert _ri["mh_overlay_eligible"]   == False, "R-I overlay should NOT be MH eligible"
assert _c1["hp_overlay_eligible"]   == True,  "C-1 should be HP eligible"
assert _ri["hp_overlay_eligible"]   == False, "R-I overlay should NOT be HP eligible"
print("mh_overlay_eligible and hp_overlay_eligible patched and validated.")

# Combine all new overlay rows
new_overlay_rows = [mh_row, hp_row, aio_row, dvao_row, tod1_row, tod2_row, fhem_row]
print(f"New overlay rows: {[r['district_code'] for r in new_overlay_rows]}")

mh_overlay_eligible and hp_overlay_eligible patched and validated.
New overlay rows: ['MH', 'HP', 'AIO', 'DVAO', 'TOD-1', 'TOD-2', 'FHEM']


## Section 7 — Assemble, validate, and export `zoning_district_standards.csv`

In [96]:
# ── Combine ALL row groups ─────────────────────────────────────────────────────
all_rows = (
    prose_rows          # S-1, RE-43, RE-24, R1-14, R-4A  (5 rows)
    + table_rows        # RE-35 through R-5                (10 rows)
    + commercial_rows   # R-O, C-O, C-1, C-2, C-3, UR     (6 rows)
    + overlay_rows      # R-I, H-R                         (2 rows)
    + new_overlay_rows  # MH, HP, AIO, DVAO, TOD-1, TOD-2, FHEM  (7 rows)
    + dtc_rows          # 15 DTC character areas           (15 rows)
    + special_rows      # PUD, WU, PC, PSC                 (4 rows)
)

df = pd.DataFrame(all_rows)

# ── Canonical column order ─────────────────────────────────────────────────────
COLUMN_ORDER = [
    # Identity
    "district_code", "district_name", "ordinance_section", "district_type",
    # Density
    "max_density_du_per_acre", "max_density_notes",
    # Height — Standard
    "max_height_ft", "max_height_stories",
    # Lot dimensions
    "min_lot_width_ft", "min_lot_depth_ft",
    # Setbacks
    "min_front_setback_ft", "min_rear_setback_ft", "min_street_side_setback_ft",
    "min_interior_side_setback_a_ft", "min_interior_side_setback_b_ft",
    # Coverage
    "max_lot_coverage_pct", "adu_coverage_bonus_pct",
    # Parking
    "parking_min_per_du", "parking_notes",
    # Use permissions
    "permits_duplex", "permits_triplex", "permits_fourplex",
    "permits_courtyard_apt", "permits_cottage_cluster", "multifamily_permitted",
    # PRD reference
    "max_density_du_per_acre_prd", "max_height_ft_prd",
    "max_height_stories_prd", "max_lot_coverage_pct_prd",
    # Flags
    "pud_flag", "wu_code_flag", "dtc_flag", "dtc_character_area",
    "mf_conversion_flag", "ri_overlay_eligible", "hr_overlay_eligible",
    "mh_overlay_eligible",
    "hp_overlay_eligible",
    # Notes
    "notes",
]

# Add any missing columns (defensive — covers edge cases in row dicts)
for col in COLUMN_ORDER:
    if col not in df.columns:
        df[col] = None

df = df[COLUMN_ORDER].copy()
print(f"Assembled: {len(df)} district rows × {len(df.columns)} columns")
print(f"\nRow counts by district_type:")
print(df["district_type"].value_counts().to_string())
df[["district_code","district_type","max_density_du_per_acre",
    "multifamily_permitted","dtc_flag","mf_conversion_flag","hr_overlay_eligible"]]

Assembled: 49 district rows × 39 columns

Row counts by district_type:
district_type
residential               16
form_based_code           16
overlay                    9
commercial                 5
planned_development        2
residential_commercial     1


,district_code,district_type,max_density_du_per_acre,multifamily_permitted,dtc_flag,mf_conversion_flag,hr_overlay_eligible
0,S-1,residential,1.00,None,False,False,False
1,RE-43,residential,1.00,None,False,False,False
2,RE-24,residential,1.80,None,False,False,False
3,R1-14,residential,3.10,None,False,False,False
4,R-4A,residential,43.56,True,False,False,True
5,RE-35,residential,1.20,False,False,False,False
6,R1-18,residential,2.00,False,False,False,False
7,R1-10,residential,3.50,False,False,False,False
8,R1-8,residential,4.50,False,False,False,False
9,R1-6,residential,5.50,False,False,False,False


In [97]:
# ── Apply parking minimums ─────────────────────────────────────────────────────
MULTIFAMILY_DISTRICTS  = {"R-2","R-3","R-3A","R-4","R-5","R-4A","C-1","C-2","C-3"}
SINGLE_FAMILY_DISTRICTS = {"S-1","RE-43","RE-24","R1-14","RE-35","R1-18","R1-10","R1-8","R1-6"}

def assign_parking(row) -> tuple:
    code = row["district_code"]
    # Already hardcoded (UR, R-O)
    if pd.notna(row.get("parking_min_per_du")):
        return row["parking_min_per_du"], row["parking_notes"]
    # Skip/placeholder rows
    if row.get("pud_flag") or row.get("wu_code_flag"):
        return None, "See applicable district or transect"
    if row.get("dtc_flag"):
        return None, "Residential: per §1206 (references §702)"
    if row.get("district_type") == "overlay":
        return None, "Per underlying base district"
    if code == "C-O":
        return None, "Accessory residential — §702.C multi-family rate applies"
    if code in MULTIFAMILY_DISTRICTS:
        return parking_multifamily, "1.5 spaces/DU; 0.25 unreserved/DU when ≥5 DUs (§702.C)"
    if code in SINGLE_FAMILY_DISTRICTS:
        return parking_sf_detached, "2 spaces per primary DU (§702.C)"
    if code == "PSC":
        return None, "Commercial use; no residential parking standard"
    return None, ""

parking_vals = [assign_parking(row) for _, row in df.iterrows()]
df["parking_min_per_du"] = [v[0] for v in parking_vals]
df["parking_notes"]      = [v[1] for v in parking_vals]

print("Parking minimums applied.")
df[["district_code","parking_min_per_du","parking_notes"]].head(25)

Parking minimums applied.


,district_code,parking_min_per_du,parking_notes
0,S-1,NaN,2 spaces per primary DU (§702.C)
1,RE-43,NaN,2 spaces per primary DU (§702.C)
2,RE-24,NaN,2 spaces per primary DU (§702.C)
3,R1-14,NaN,2 spaces per primary DU (§702.C)
4,R-4A,1.5,1.5 spaces/DU; 0.25 unreserved/DU when ≥5 DUs ...
5,RE-35,NaN,2 spaces per primary DU (§702.C)
6,R1-18,NaN,2 spaces per primary DU (§702.C)
7,R1-10,NaN,2 spaces per primary DU (§702.C)
8,R1-8,NaN,2 spaces per primary DU (§702.C)
9,R1-6,NaN,2 spaces per primary DU (§702.C)


In [98]:
# ── Validation ────────────────────────────────────────────────────────────────
errors = []

# 1. No duplicate district codes
dupes = df[df.duplicated("district_code", keep=False)]["district_code"].unique()
if len(dupes):
    errors.append(f"Duplicate district_code(s): {list(dupes)}")

# 2. All district_type values are expected
VALID_TYPES = {"residential","commercial","residential_commercial",
               "planned_development","form_based_code","overlay"}
bad_types = df[~df["district_type"].isin(VALID_TYPES)]["district_type"].unique()
if len(bad_types):
    errors.append(f"Unexpected district_type(s): {list(bad_types)}")

# 3. Residential MF-permitted rows (non-DTC/WU/overlay/UR) must have density > 0
mf_no_density = df[
    (df["multifamily_permitted"] == True) &
    df["max_density_du_per_acre"].isna() &
    (~df["dtc_flag"]) &
    (~df["wu_code_flag"]) &
    (df["district_type"] != "overlay") &
    (~df["district_code"].isin(["UR","R-O"]))
]["district_code"].tolist()
if mf_no_density:
    errors.append(f"MF permitted but density NULL (unexpected): {mf_no_density}")

# 4. Single-family districts should have permits_* = False, not None
SF_CODES = ["S-1","RE-43","RE-24","R1-14","RE-35","R1-18","R1-10","R1-8","R1-6"]
for col in ["permits_duplex","permits_triplex","permits_fourplex"]:
    null_sf = df[(df["district_code"].isin(SF_CODES)) & df[col].isna()]["district_code"].tolist()
    if null_sf:
        errors.append(f"{col} is NULL (should be False) for: {null_sf}")

# 5. DTC rows must have dtc_flag=True and dtc_character_area set
dtc_check = df[df["district_code"].str.startswith("DTC-", na=False)]
dtc_missing = dtc_check[dtc_check["dtc_character_area"].isna()]["district_code"].tolist()
if dtc_missing:
    errors.append(f"DTC rows missing dtc_character_area: {dtc_missing}")

# 6. Exactly 15 DTC rows
dtc_count = len(dtc_check)
if dtc_count != 15:
    errors.append(f"Expected 15 DTC rows, found {dtc_count}")

# 7. All columns present
missing_cols = [c for c in COLUMN_ORDER if c not in df.columns]
if missing_cols:
    errors.append(f"Missing columns in output: {missing_cols}")

# 8. WU row has wu_code_flag = True
if not df[df["district_code"] == "WU"]["wu_code_flag"].any():
    errors.append("WU row missing wu_code_flag = True")

# ── Summary ────────────────────────────────────────────────────────────────────
if errors:
    print(f"\n⚠️  {len(errors)} validation error(s):")
    for e in errors:
        print(f"  • {e}")
else:
    print(f"✅ Validation passed — {len(df)} rows, {len(df.columns)} columns, 0 errors.")

print(f"\nExpected row count: 5 prose + 10 table + 6 commercial + 2 overlay + 15 DTC + 4 special = 42")
print(f"Actual row count:   {len(df)}")


⚠️  3 validation error(s):
  • permits_duplex is NULL (should be False) for: ['S-1', 'RE-43', 'RE-24', 'R1-14']
  • permits_triplex is NULL (should be False) for: ['S-1', 'RE-43', 'RE-24', 'R1-14']
  • permits_fourplex is NULL (should be False) for: ['S-1', 'RE-43', 'RE-24', 'R1-14']

Expected row count: 5 prose + 10 table + 6 commercial + 2 overlay + 15 DTC + 4 special = 42
Actual row count:   49


In [99]:
# ── Export zoning_district_standards.csv ─────────────────────────────────────
OUTPUT_CSV.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUTPUT_CSV, index=False)
print(f"✅ Exported: {OUTPUT_CSV}")
print(f"   {len(df)} rows × {len(df.columns)} columns")
print(f"   File size: {OUTPUT_CSV.stat().st_size / 1024:.1f} KB")

✅ Exported: D:\DevSource\phoenix-zoning-scanner\data\reference\zoning_district_standards.csv
   49 rows × 39 columns
   File size: 32.0 KB


## Section 8 — WU Code transect standards → `wu_transect_standards.csv`

The Walkable Urban Code (Chapter 13) applies within approved TOD district boundaries
per the §1301.C applicability map. Twelve transect districts plus a Special Districts (SD) option.

**Data sources:**
- **Table 1303.2** — lot standards per transect (height, setback, coverage, frontage %)
- **Table 1306.1** — land use matrix (SFD/SFA/MF/Live-Work by transect)
- **Table 1307.1** — parking minimums by transect
- **§1303.A.2** — adjacency stepback: 15-ft max height within 10 ft of SF zone
- **§1303.A.3** — tall building setbacks from R1-10/R1-8/R1-6/HP zones

**`primary_frontage_is_max`** = `True` means the setback is a build-to line (maximum);
`False` means it is a minimum. This distinction is critical for form-based code scoring.

**Important parking note (Table 1307.1, footnote 1):**
An additional 50% reduction applies when parking is within 1,320 ft of a light rail station.
This is encoded in the `notes` field; implement as a spatial condition in the scanner.

In [100]:
# ── WU Code transect data ────────────────────────────────────────────────────
# Source: Tables 1303.2, 1306.1, 1307.1 and §1303.A.2–3 (Chapter 13 HTML)

_WU_TRANSECTS = [
    # (code, name, height_ft, front_sb_ft, front_is_max,
    #  sec_sb_ft, side_sb_ft, rear_sb_ft,
    #  lot_cov_pct, min_bldg_frontage_pct,
    #  park_mf, park_sfd,
    #  permits_sfd, permits_sfa, permits_mf, permits_live_work,
    #  notes)
    (
        "T3:2", "T3 Low-Intensity Residential", 30,
        20, False,                      # 20-ft MINIMUM primary frontage
        12, 3, 20,                      # secondary, side (asymmetric 3+10), rear
        60, 50,                         # lot coverage max, min building frontage %
        None, 2.0,                      # parking MF (NP), SFD
        True, True, False, True,
        "Low-intensity residential (Table 1303.2). "
        "SFD and SFA permitted; MF not permitted (Table 1306.1). "
        "Primary frontage setback: 20-ft MINIMUM. Side: 3 ft + 10 ft asymmetric. "
        "Stepback: 15-ft max height within 10 ft of adjacent SF zone (§1303.A.2). "
        "MF parking N/A. SFD: 2.0 spaces/unit (Table 1307.1).",
    ),
    (
        "T4:2", "T4:2 Low-Intensity Urban Residential", 30,
        20, True,                       # 20-ft MAXIMUM (build-to line)
        8, 0, 0,
        70, 60,
        0.75, None,
        False, True, True, True,
        "Low-intensity urban residential (Table 1303.2). Max 30 ft. "
        "SFD not permitted; SFA and MF permitted (Table 1306.1). "
        "Primary frontage: 20-ft MAXIMUM (build-to). "
        "Side/rear: 0-ft min with adjacency rules per §1303.A.2 when near SF/HP zones. "
        "MF parking: 0.75 spaces/unit; 50% additional reduction within 1,320 ft of LRT (Table 1307.1 fn.1).",
    ),
    (
        "T4:3", "T4:3 Low-Intensity Urban Residential", 40,
        20, True,
        8, 0, 0,
        70, 60,
        0.75, None,
        False, True, True, True,
        "Low-intensity urban residential (Table 1303.2). Max 40 ft. "
        "SFD not permitted; SFA and MF permitted. "
        "SFI subdivision: max 48 ft per §1303.A.1.a. "
        "Adjacency stepback per §1303.A.2. "
        "MF parking: 0.75/unit; LRT reduction applicable.",
    ),
    (
        "T5:2", "T5:2 Low-Intensity Mixed-Use", 30,
        12, True,
        10, 0, 0,
        80, 70,
        0.75, None,
        False, True, True, True,
        "Low-intensity mixed-use (Table 1303.2). Max 30 ft. "
        "Ground floor commercial frontage: 75% glazing required. "
        "Adjacency stepback at 48 ft: 20-ft stepback from SF/HP zones (§1303.A.2). "
        "MF parking: 0.75/unit; LRT reduction applicable.",
    ),
    (
        "T5:3", "T5:3 Low-Intensity Mixed-Use", 48,
        12, True,
        10, 0, 0,
        80, 70,
        0.75, None,
        False, True, True, True,
        "Low-intensity mixed-use (Table 1303.2). Max 48 ft. "
        "At 48 ft: 20-ft stepback required adjacent to SF or HP zones (§1303.A.2). "
        "MF parking: 0.75/unit; LRT reduction applicable.",
    ),
    (
        "T5:5", "T5:5 Medium-High Mixed-Use", 56,
        12, True,
        10, 0, 0,
        80, 70,
        0.75, None,
        False, True, True, True,
        "Medium-high mixed-use (Table 1303.2). Max 56 ft. "
        "At 48 ft: 20-ft stepback adjacent to SF/HP (§1303.A.2). "
        "Parking structures: cannot exceed building height; 20-ft stepback from SF/HP if >40 ft. "
        "MF parking: 0.75/unit; LRT reduction applicable.",
    ),
    (
        "T5:6", "T5:6 Medium-High Mixed-Use", 80,
        12, True,
        10, 0, 0,
        80, 70,
        0.75, None,
        False, True, True, True,
        "Medium-high mixed-use (Table 1303.2). Max 80 ft. "
        "Buildings ≥48 ft adjacent to R1-10/R1-8/R1-6/HP (§1303.A.3): "
        "25-ft setback (40–80 ft height); 30-ft + 1 ft per 2 ft above 80 ft, max 75-ft setback. "
        "MF parking: 0.75/unit; LRT reduction applicable.",
    ),
    (
        "T5:7", "T5:7 Medium-High Mixed-Use", 100,
        12, True,
        10, 0, 0,
        80, 70,
        0.75, None,
        False, True, True, True,
        "Medium-high mixed-use (Table 1303.2). Max 100 ft. "
        "Buildings ≥48 ft adjacent to R1-10/R1-8/R1-6/HP: 25-ft setback (40–80 ft); "
        "30-ft + 1 ft per 2 ft above 80 ft, max 75-ft setback (§1303.A.3). "
        "MF parking: 0.75/unit; LRT reduction applicable.",
    ),
    (
        "T6:7", "T6:7 High-Intensity Mixed-Use", 100,
        12, True,
        10, 0, 0,
        90, 80,
        0.75, None,
        False, True, True, True,
        "High-intensity mixed-use (Table 1303.2). Max 100 ft. "
        "At 48 ft: 20-ft stepback adjacent to SF/HP zones. "
        "Secondary building frontage min: 70%. "
        "MF parking: 0.75/unit (T6, ≤5 stories); further LRT reduction applicable (Table 1307.1 fn.1).",
    ),
    (
        "T6:15", "T6:15 High-Intensity Mixed-Use", 180,
        12, True,
        10, 0, 0,
        90, 80,
        0.75, None,
        False, True, True, True,
        "High-intensity mixed-use (Table 1303.2). Max 180 ft. "
        "At 48 ft: 20-ft stepback from SF/HP. "
        "Buildings ≥48 ft adjacent to R1-10/R1-8/R1-6/HP: §1303.A.3 setback rules. "
        "MF parking: 0.75/unit; LRT reduction applicable.",
    ),
    (
        "T6:22", "T6:22 High-Intensity Mixed-Use", 250,
        12, True,
        10, 0, 0,
        90, 80,
        0.75, None,
        False, True, True, True,
        "High-intensity mixed-use (Table 1303.2). Max 250 ft. "
        "At 48 ft: 20-ft stepback from SF/HP. "
        "At 120 ft: additional stepback requirement from SF/HP neighborhoods. "
        "Buildings ≥48 ft adjacent to R1-10/R1-8/R1-6/HP: 25-ft setback (40–80 ft); "
        "30-ft + 1 ft per 2 ft above 80 ft, max 75-ft setback (§1303.A.3). "
        "MF parking: 0.75/unit; LRT reduction applicable.",
    ),
    (
        "T6:HWR", "T6:HWR Height Waiver High-Intensity", None,
        12, True,
        10, 0, 0,
        90, 80,
        0.75, None,
        False, True, True, True,
        "Height waiver high-intensity (Table 1303.2). No maximum height — "
        "per existing and new height waiver entitlements beyond T6:22. "
        "All other T6 standards apply as baseline. "
        "MF parking: 0.75/unit; LRT reduction applicable.",
    ),
    (
        "SD", "Special Districts", None,
        None, None,
        None, None, None,
        None, None,
        None, None,
        None, None, None, None,
        "Special Districts — assigned to sites that cannot conform to any transect: "
        "hospitals, colleges, civic spaces, airports, transportation facilities (§1302.A.7). "
        "Standards governed by site-specific entitlements or special plans. "
        "All fields NULL.",
    ),
]

# ── Build DataFrame ────────────────────────────────────────────────────────────
wu_cols = [
    "transect_code", "transect_name",
    "max_height_ft",
    "primary_frontage_setback_ft", "primary_frontage_is_max",
    "secondary_frontage_setback_ft",
    "min_side_setback_ft", "min_rear_setback_ft",
    "max_lot_coverage_pct", "min_primary_building_frontage_pct",
    "parking_min_per_du_mf", "parking_min_sfd",
    "permits_sfd", "permits_sfa", "permits_mf", "permits_live_work",
    "notes",
]

wu_rows = []
for t in _WU_TRANSECTS:
    (code, name, height, front_sb, front_is_max, sec_sb, side_sb, rear_sb,
     lot_cov, min_front, park_mf, park_sfd, sfd, sfa, mf, lw, note) = t
    wu_rows.append({
        "transect_code":                     code,
        "transect_name":                     name,
        "max_height_ft":                     height,
        "primary_frontage_setback_ft":       front_sb,
        "primary_frontage_is_max":           front_is_max,
        "secondary_frontage_setback_ft":     sec_sb,
        "min_side_setback_ft":               side_sb,
        "min_rear_setback_ft":               rear_sb,
        "max_lot_coverage_pct":              lot_cov,
        "min_primary_building_frontage_pct": min_front,
        "parking_min_per_du_mf":             park_mf,
        "parking_min_sfd":                   park_sfd,
        "permits_sfd":                       sfd,
        "permits_sfa":                       sfa,
        "permits_mf":                        mf,
        "permits_live_work":                 lw,
        "notes":                             note,
    })

wu_df = pd.DataFrame(wu_rows)[wu_cols]
print(f"Built wu_transect_standards: {len(wu_df)} rows × {len(wu_df.columns)} columns")
wu_df[["transect_code","max_height_ft","max_lot_coverage_pct",
       "permits_sfd","permits_mf","parking_min_per_du_mf"]]

Built wu_transect_standards: 13 rows × 17 columns


,transect_code,max_height_ft,max_lot_coverage_pct,permits_sfd,permits_mf,parking_min_per_du_mf
0,T3:2,30.0,60.0,True,False,NaN
1,T4:2,30.0,70.0,False,True,0.75
2,T4:3,40.0,70.0,False,True,0.75
3,T5:2,30.0,80.0,False,True,0.75
4,T5:3,48.0,80.0,False,True,0.75
5,T5:5,56.0,80.0,False,True,0.75
6,T5:6,80.0,80.0,False,True,0.75
7,T5:7,100.0,80.0,False,True,0.75
8,T6:7,100.0,90.0,False,True,0.75
9,T6:15,180.0,90.0,False,True,0.75


In [101]:
# ── WU transect validation ────────────────────────────────────────────────────
wu_errors = []

t3 = wu_df[wu_df["transect_code"] == "T3:2"].iloc[0]
if t3["permits_mf"]:
    wu_errors.append("T3:2 should not permit MF")

t4_plus = wu_df[wu_df["transect_code"].str.startswith("T", na=False) &
                ~wu_df["transect_code"].str.startswith("T3", na=False)]
mf_issues = t4_plus[t4_plus["permits_mf"] != True]["transect_code"].tolist()
if mf_issues:
    wu_errors.append(f"T4+ transects missing MF permission: {mf_issues}")

sfd_non_t3 = wu_df[(wu_df["permits_sfd"] == True) &
                   (wu_df["transect_code"] != "T3:2")]["transect_code"].tolist()
if sfd_non_t3:
    wu_errors.append(f"SFD permitted in unexpected transects: {sfd_non_t3}")

park_issues = t4_plus[
    ~t4_plus["transect_code"].isin(["SD"]) &
    (t4_plus["parking_min_per_du_mf"] != 0.75)
]["transect_code"].tolist()
if park_issues:
    wu_errors.append(f"Unexpected parking_min_per_du_mf in: {park_issues}")

if wu_errors:
    print(f"⚠️  {len(wu_errors)} WU validation error(s):")
    for e in wu_errors:
        print(f"  • {e}")
else:
    print(f"✅ WU transect validation passed — {len(wu_df)} rows, 0 errors.")

# ── Export ────────────────────────────────────────────────────────────────────
OUTPUT_WU_CSV.parent.mkdir(parents=True, exist_ok=True)
wu_df.to_csv(OUTPUT_WU_CSV, index=False)
print(f"\n✅ Exported: {OUTPUT_WU_CSV}")
print(f"   {len(wu_df)} rows × {len(wu_df.columns)} columns")
print(f"   File size: {OUTPUT_WU_CSV.stat().st_size / 1024:.1f} KB")

✅ WU transect validation passed — 13 rows, 0 errors.

✅ Exported: D:\DevSource\phoenix-zoning-scanner\data\reference\wu_transect_standards.csv
   13 rows × 17 columns
   File size: 4.8 KB


## Post-export checklist

**`zoning_district_standards.csv` (49 rows expected):**
- [ ] 5 prose + 10 table + 6 commercial + 2 overlay + 7 new_overlay + 15 DTC + 4 special = **49 rows**
- [ ] R1-10/R1-8/R1-6: `max_height_stories_prd = 3`, `max_height_ft_prd = 30`
- [ ] R-2 through R-5: `adu_coverage_bonus_pct = 10.0`
- [ ] R-4A: `ri_overlay_eligible = True`, `hr_overlay_eligible = True`
- [ ] C-1/C-2/C-3: `max_density_du_per_acre = 14.5`, `max_height_ft = 30.0`, `mf_conversion_flag = True`
- [ ] UR: `max_height_ft = 75.0`, `max_density_du_per_acre = None`, `max_lot_coverage_pct = None`
- [ ] R-O: `max_height_ft = None` — add manual override if city records available
- [ ] All 15 DTC rows: `dtc_flag = True`, `max_height_ft = None`, `multifamily_permitted = True`
- [ ] MH row: `permits_fourplex = True`, `parking_min_per_du = 1.0`
- [ ] HP row: all numeric standards NULL; `hp_overlay_eligible = True`
- [ ] AIO row: `mh_overlay_eligible = False` (excluded near Sky Harbor)
- [ ] DVAO row: `mh_overlay_eligible = False` (excluded near Deer Valley Airport)
- [ ] TOD-1/TOD-2: `mf_conversion_flag = True`, `mh_overlay_eligible = True`
- [ ] FHEM row: `multifamily_permitted = False`, `max_lot_coverage_pct = 0.0`
- [ ] `mh_overlay_eligible = True` for: RE-43, RE-24, R1-14, RE-35 through R-4A, C-1, C-2, C-3
- [ ] `mh_overlay_eligible = False` for: S-1, R-O, C-O, UR, overlay rows, DTC rows, special rows
- [ ] `hp_overlay_eligible = True` for: all residential and commercial base districts
- [ ] `hp_overlay_eligible = False` for: overlay rows, DTC rows, WU rows, special rows

**`wu_transect_standards.csv` (13 rows expected):**
- [ ] T3:2: `permits_mf = False`, `parking_min_per_du_mf = None`, `primary_frontage_is_max = False`
- [ ] T4:2: `permits_mf = True`, `max_height_ft = 30`, `parking_min_per_du_mf = 0.75`
- [ ] T6:22: `max_height_ft = 250`, `max_lot_coverage_pct = 90`
- [ ] T6:HWR: `max_height_ft = None`
- [ ] SD: all numeric fields `None`

**Overlays NOT encoded as rows (geographic — join via supplementary zoning maps):**
- §644 AIO boundary: Supplementary Zoning Map (Sky Harbor vicinity)
- §658 DVAO boundary: Supplementary Zoning Map No. 1116
- §662 TOD-1 boundary: Supplementary Zoning Map No. 1086
- §663 TOD-2 boundary: Supplementary Zoning Map No. 1086
- §657 FHEM: applied via FCDMC Master Plan Study area
- §653 Desert Character Overlay: north Phoenix only
- HP designations: Phoenix Historic Property Register

**⚠️ Missing from ordinance HTML (not codeable from this source):**
- §625 DOES NOT EXIST — gap in the ordinance between §624 and §626
- No "Hillside" overlay district exists in the Phoenix ZO (check General Plan hillside policies)
- No "Freeway Frontage District" as a standalone overlay code
- DTC height/density: must come from §1202.B–C GIS layers (maps, not text)

**DuckDB integration:**
```python
# Load both CSVs into scanner database
con.execute("CREATE TABLE zoning_district_standards AS SELECT * FROM read_csv_auto('data/reference/zoning_district_standards.csv')")
con.execute("CREATE TABLE wu_transect_standards AS SELECT * FROM read_csv_auto('data/reference/wu_transect_standards.csv')")

# Example: find MH-eligible parcels outside airport noise zones
# SELECT p.parcel_id, z.district_code, z.mh_overlay_eligible
# FROM parcels p
# JOIN zoning_district_standards z ON p.zone_code = z.district_code
# WHERE z.mh_overlay_eligible = TRUE
#   AND p.parcel_id NOT IN (SELECT parcel_id FROM parcels_aio)  -- spatial join to AIO boundary
#   AND p.parcel_id NOT IN (SELECT parcel_id FROM parcels_dvao) -- spatial join to DVAO boundary
```